# Bloc 7 — Advanced Model Refinement & Adaptive Forecasting
# Perfectionnement avancé, régimes d'erreur et adaptation du modèle

In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# ============================================================
# DONNEES
# ============================================================

X = joblib.load(
    "data_processed/X_block3_weather_nonlinear.joblib"
)

y = joblib.load(
    "data_processed/y_block3_weather.joblib"
)

# ============================================================
# RESULTATS FINAUX BLOC 6
# ============================================================

state = joblib.load(
    "models/block6_final_state.joblib"
)

idx_2024 = state["val_index_2024"]
y_2024 = state["y_2024"]
pred_2024 = state["pred_patch_2024"]

idx_2025 = state["eval_index_2025"]
y_2025 = state["y_2025"]
pred_2025 = state["pred_patch_2025"]

# Résidus du champion
residual_2024 = (
    y_2024 - pred_2024
)

residual_2025 = (
    y_2025 - pred_2025
)

print(
    "2024 :",
    len(y_2024)
)

print(
    "2025 :",
    len(y_2025)
)

print(
    "RMSE 2025 :",
    np.sqrt(
        np.mean(
            residual_2025**2
        )
    )
)

2024 : 4416
2025 : 17517
RMSE 2025 : 177.58245670759058


## Étape 1 — Residual Regime Audit
### Identification des situations encore difficiles à prévoir

### Partie A — Mise en pratique

In [2]:
residual_df = pd.DataFrame(
    {
        "y_true":
            y_2025,

        "prediction":
            pred_2025,

        "residual":
            residual_2025
    },
    index=idx_2025
)

residual_df["abs_error"] = (
    residual_df["residual"]
    .abs()
)

residual_df["sq_error"] = (
    residual_df["residual"] ** 2
)

In [3]:
print(
    "Mean residual :",
    residual_df["residual"].mean()
)

print(
    "Residual std :",
    residual_df["residual"].std()
)

print(
    "RMSE :",
    np.sqrt(
        residual_df[
            "sq_error"
        ].mean()
    )
)

print(
    "MAE :",
    residual_df[
        "abs_error"
    ].mean()
)

print(
    "Median abs error :",
    residual_df[
        "abs_error"
    ].median()
)

print(
    "95% abs error :",
    residual_df[
        "abs_error"
    ].quantile(0.95)
)

print(
    "99% abs error :",
    residual_df[
        "abs_error"
    ].quantile(0.99)
)

print(
    "Maximum error :",
    residual_df[
        "abs_error"
    ].max()
)

Mean residual : -3.26416024799284
Residual std : 177.5575229722794
RMSE : 177.58245670759058
MAE : 128.24532895554032
Median abs error : 99.8880946301797
95% abs error : 330.87095676867176
99% abs error : 525.3731811783913
Maximum error : 2847.222015382489


In [4]:
residual_df["hour"] = (
    residual_df.index.hour
)

residual_df["dow"] = (
    residual_df.index.dayofweek
)

residual_df["month"] = (
    residual_df.index.month
)

In [5]:
def grouped_error_table(
    df,
    column
):

    result = (
        df.groupby(column)
        .agg(
            n=("residual", "size"),

            bias=(
                "residual",
                "mean"
            ),

            mae=(
                "abs_error",
                "mean"
            ),

            mse=(
                "sq_error",
                "mean"
            )
        )
    )

    result["rmse"] = np.sqrt(
        result["mse"]
    )

    return (
        result
        .drop(columns="mse")
    )

In [6]:
hour_errors = grouped_error_table(
    residual_df,
    "hour"
)

dow_errors = grouped_error_table(
    residual_df,
    "dow"
)

month_errors = grouped_error_table(
    residual_df,
    "month"
)

print("\nWorst hours:")
print(
    hour_errors
    .sort_values(
        "rmse",
        ascending=False
    )
    .head()
)

print("\nWorst weekdays:")
print(
    dow_errors
    .sort_values(
        "rmse",
        ascending=False
    )
)

print("\nWorst months:")
print(
    month_errors
    .sort_values(
        "rmse",
        ascending=False
    )
)


Worst hours:
        n       bias         mae        rmse
hour                                        
11    730  10.284274  146.953800  226.022242
12    730   9.829389  152.768440  221.989855
18    730 -14.324642  149.478171  213.275124
17    730 -30.802250  151.182942  196.252609
13    730 -25.139147  147.641471  196.155565

Worst weekdays:
        n       bias         mae        rmse
dow                                         
6    2496 -13.563689  138.002654  225.675170
0    2496  -6.933118  140.889492  192.541441
3    2496   0.801560  127.026397  170.253753
5    2496   0.624945  130.437013  167.276908
1    2496   2.110100  124.706405  165.064744
2    2541   2.347192  119.997714  159.182788
4    2496  -8.337279  116.806323  152.661368

Worst months:
          n       bias         mae        rmse
month                                         
10     1488  -0.511012  149.718477  241.424241
3      1488  -9.231122  150.680439  238.601583
1      1488  -4.496110  146.180799  196.228746

In [7]:
residual_df["load_decile"] = (
    pd.qcut(
        residual_df["y_true"],
        q=10,
        labels=False,
        duplicates="drop"
    )
)

In [8]:
load_errors = grouped_error_table(
    residual_df,
    "load_decile"
)

print(
    load_errors
)

                n       bias         mae        rmse
load_decile                                         
0            1753  -9.075811   96.873280  126.352672
1            1751   1.070729  113.273222  163.957594
2            1751   7.834744  120.597284  159.711530
3            1753  -9.215542  126.659347  175.923719
4            1751 -10.625340  128.646435  186.955142
5            1751   1.488394  144.002082  220.575232
6            1752  -3.850969  128.743636  172.251327
7            1752  -7.931927  142.395752  186.982315
8            1751  -0.413291  135.948563  176.565758
9            1752  -1.906928  145.333208  191.446810


In [9]:
temperature_candidates = [
    "temp_lag_1h",
    "temp_mean_24h",
    "temp_lag_24h"
]

temperature_feature = next(
    (
        c
        for c in temperature_candidates
        if c in X.columns
    ),
    None
)

print(
    "Temperature feature :",
    temperature_feature
)

Temperature feature : temp_lag_1h


In [10]:
if temperature_feature is not None:

    residual_df["temperature"] = (
        X.loc[
            residual_df.index,
            temperature_feature
        ]
        .to_numpy()
    )

    residual_df["temp_group"] = (
        pd.qcut(
            residual_df["temperature"],
            q=5,
            labels=[
                "Very cold",
                "Cold",
                "Mild",
                "Warm",
                "Hot"
            ]
        )
    )

    temp_errors = grouped_error_table(
        residual_df,
        "temp_group"
    )

    print(
        temp_errors
    )

               n      bias         mae        rmse
temp_group                                        
Very cold   3507 -7.776148  137.046056  179.798014
Cold        3510 -3.796877  141.124688  213.577312
Mild        3494  4.340683  134.551573  190.216335
Warm        3506 -1.348358  114.881848  149.868371
Hot         3500 -7.719803  113.601799  145.168795


In [11]:
residual_df["load_change"] = (
    residual_df[
        "y_true"
    ].diff()
)

residual_df["abs_load_change"] = (
    residual_df[
        "load_change"
    ].abs()
)

In [12]:
residual_df["ramp_group"] = (
    pd.qcut(
        residual_df[
            "abs_load_change"
        ],
        q=5,
        labels=[
            "Very stable",
            "Stable",
            "Moderate",
            "Fast",
            "Extreme"
        ],
        duplicates="drop"
    )
)

ramp_errors = grouped_error_table(
    residual_df.dropna(),
    "ramp_group"
)

print(
    ramp_errors
)

                n       bias         mae        rmse
ramp_group                                          
Very stable  3504  -8.038302  129.945551  174.389361
Stable       3505  -4.834282  124.814561  171.785249
Moderate     3513 -12.526879  116.810815  155.733687
Fast         3498  -2.309442  127.724916  174.958356
Extreme      3496  11.447692  142.027268  207.232142


In [13]:
lags = {
    "30 min": 1,
    "1 hour": 2,
    "1 day": 48,
    "1 week": 336
}

for name, lag in lags.items():

    corr = np.corrcoef(
        residual_2025[:-lag],
        residual_2025[lag:]
    )[0, 1]

    print(
        name,
        ":",
        corr
    )

30 min : 0.07845949453850126
1 hour : -0.005105623949911672
1 day : 0.036097045988930224
1 week : 0.006300805615806612


In [14]:
threshold_99 = (
    residual_df[
        "abs_error"
    ].quantile(0.99)
)

extreme_errors = residual_df[
    residual_df[
        "abs_error"
    ]
    >= threshold_99
].copy()

print(
    "Threshold 99% :",
    threshold_99
)

print(
    "Extreme observations :",
    len(extreme_errors)
)

print(
    extreme_errors[
        [
            "y_true",
            "prediction",
            "residual",
            "abs_error",
            "hour",
            "dow",
            "month"
        ]
    ]
    .sort_values(
        "abs_error",
        ascending=False
    )
    .head(15)
)

Threshold 99% : 525.3731811783913
Extreme observations : 176
                            y_true    prediction     residual    abs_error  \
2025-03-30 11:30:00+00:00  50629.0  53476.222015 -2847.222015  2847.222015   
2025-10-26 02:00:00+00:00  38716.0  41535.916275 -2819.916275  2819.916275   
2025-10-26 11:30:00+00:00  51274.0  48887.883317  2386.116683  2386.116683   
2025-03-30 21:30:00+00:00  49678.0  51541.051013 -1863.051013  1863.051013   
2025-10-27 00:00:00+00:00  45302.0  47045.807046 -1743.807046  1743.807046   
2025-10-26 21:30:00+00:00  48062.0  46398.867429  1663.132571  1663.132571   
2025-10-26 12:30:00+00:00  49711.0  51317.408772 -1606.408772  1606.408772   
2025-03-30 18:00:00+00:00  47472.0  49066.830356 -1594.830356  1594.830356   
2025-03-30 12:00:00+00:00  48446.0  50010.613948 -1564.613948  1564.613948   
2025-03-30 23:00:00+00:00  46061.0  47581.182263 -1520.182263  1520.182263   
2025-03-30 12:30:00+00:00  47038.0  45581.538450  1456.461550  1456.461550   
202

In [15]:
audit_results = {
    "global":
        residual_df[
            [
                "residual",
                "abs_error"
            ]
        ].describe(),

    "hour":
        hour_errors,

    "dow":
        dow_errors,

    "month":
        month_errors,

    "load":
        load_errors,

    "ramp":
        ramp_errors,

    "extreme_errors":
        extreme_errors
}

if temperature_feature is not None:

    audit_results[
        "temperature"
    ] = temp_errors


joblib.dump(
    audit_results,
    "models/block7_residual_audit.joblib"
)

joblib.dump(
    residual_df,
    "data_processed/block7_residuals_2025.joblib"
)

print(
    "Audit sauvegardé."
)

Audit sauvegardé.


### Partie C — Analyse

Lorsque la performance globale d'un modèle devient très élevée, une amélioration uniforme devient difficile.

Il est alors plus pertinent d'étudier la perte conditionnelle :

$$
E[L(Y,\hat Y)\mid X\in R]
$$

pour différents régimes $R$.

Un modèle peut être excellent globalement tout en restant moins précis pendant certaines heures, certaines saisons, certaines températures ou certains niveaux de consommation.

L'objectif du Bloc 7 est donc d'identifier ces régimes difficiles avant de construire des correcteurs spécialisés.

Cette approche permettra notamment d'étudier :

- les modèles spécialisés ;
- les Mixture-of-Experts ;
- le concept drift ;
- l'apprentissage adaptatif ;
- la significativité statistique des gains.

## Étape 2 — DST / Local-Time Regime Audit

### Partie A — Mise en pratique

In [16]:
# ============================================================
# CHANGEMENTS D'HEURE OFFICIELS
# Heure UTC
# ============================================================

dst_2024 = [
    pd.Timestamp(
        "2024-03-31 01:00:00",
        tz="UTC"
    ),

    pd.Timestamp(
        "2024-10-27 01:00:00",
        tz="UTC"
    )
]

dst_2025 = [
    pd.Timestamp(
        "2025-03-30 01:00:00",
        tz="UTC"
    ),

    pd.Timestamp(
        "2025-10-26 01:00:00",
        tz="UTC"
    )
]

In [17]:
def analyse_dst_window(
    index,
    y_true,
    prediction,
    transitions,
    window_hours=24
):

    index = pd.DatetimeIndex(index)

    residual = (
        np.asarray(y_true)
        - np.asarray(prediction)
    )

    # ----------------------------------------
    # Distance au changement d'heure
    # ----------------------------------------

    min_distance = np.full(
        len(index),
        np.inf
    )

    transition_id = np.full(
        len(index),
        -1
    )

    for i, transition in enumerate(
        transitions
    ):

        distance = (
            (
                index
                - transition
            )
            / pd.Timedelta(
                hours=1
            )
        ).to_numpy()

        better = (
            np.abs(distance)
            <
            np.abs(min_distance)
        )

        min_distance[better] = (
            distance[better]
        )

        transition_id[better] = i


    # ----------------------------------------
    # Fenêtre autour du changement d'heure
    # ----------------------------------------

    dst_mask = (
        np.abs(min_distance)
        <= window_hours
    )


    df = pd.DataFrame(
        {
            "residual":
                residual,

            "sq_error":
                residual**2,

            "abs_error":
                np.abs(residual),

            "hours_to_dst":
                min_distance,

            "dst_window":
                dst_mask,

            "transition":
                transition_id
        },
        index=index
    )


    # ----------------------------------------
    # Résumé
    # ----------------------------------------

    rows = []

    for name, mask in [
        ("DST window", dst_mask),
        ("Normal", ~dst_mask)
    ]:

        r = residual[
            mask
        ]

        rows.append(
            {
                "Regime":
                    name,

                "n":
                    len(r),

                "Bias":
                    np.mean(r),

                "MAE":
                    np.mean(
                        np.abs(r)
                    ),

                "RMSE":
                    np.sqrt(
                        np.mean(
                            r**2
                        )
                    )
            }
        )


    summary = pd.DataFrame(
        rows
    )

    total_sse = np.sum(
        residual**2
    )

    dst_sse = np.sum(
        residual[
            dst_mask
        ]**2
    )

    sse_share = (
        100
        * dst_sse
        / total_sse
    )


    return (
        df,
        summary,
        sse_share
    )

In [18]:
dst_df_2025, dst_summary_2025, dst_sse_share_2025 = (
    analyse_dst_window(
        idx_2025,
        y_2025,
        pred_2025,
        dst_2025,
        window_hours=24
    )
)

print(
    dst_summary_2025
)

print(
    "\nPart du SSE provenant des fenêtres DST :",
    dst_sse_share_2025,
    "%"
)

       Regime      n       Bias         MAE        RMSE
0  DST window    194 -31.377469  383.173511  630.014031
1      Normal  17323  -2.949320  125.390392  165.661160

Part du SSE provenant des fenêtres DST : 13.939343604113043 %


In [19]:
dst_df_2024, dst_summary_2024, dst_sse_share_2024 = (
    analyse_dst_window(
        idx_2024,
        y_2024,
        pred_2024,
        dst_2024,
        window_hours=24
    )
)

print(
    dst_summary_2024
)

print(
    "\nPart du SSE DST 2024 :",
    dst_sse_share_2024,
    "%"
)

       Regime     n       Bias         MAE        RMSE
0  DST window    97  23.389847  394.944923  669.245536
1      Normal  4319  -6.575128  131.846886  177.141234

Part du SSE DST 2024 : 24.275008854846867 %


In [20]:
window_results = []

for h in [
    3,
    6,
    12,
    24,
    36,
    48
]:

    _, summary, share = (
        analyse_dst_window(
            idx_2025,
            y_2025,
            pred_2025,
            dst_2025,
            window_hours=h
        )
    )

    dst_row = (
        summary
        .loc[
            summary["Regime"]
            == "DST window"
        ]
        .iloc[0]
    )

    window_results.append(
        {
            "Window_hours":
                h,

            "n":
                dst_row["n"],

            "RMSE_DST":
                dst_row["RMSE"],

            "MAE_DST":
                dst_row["MAE"],

            "SSE_share_%":
                share
        }
    )


window_results = pd.DataFrame(
    window_results
)

print(
    window_results
)

   Window_hours    n    RMSE_DST     MAE_DST  SSE_share_%
0             3   26  729.561181  434.995949     2.505168
1             6   50  550.105633  316.511186     2.739065
2            12   98  636.123088  352.541230     7.178745
3            24  194  630.014031  383.173511    13.939344
4            36  290  541.700571  326.012395    15.404824
5            48  386  505.749298  309.495871    17.873024


In [21]:
local_index = (
    residual_df.index
    .tz_convert(
        "Europe/Paris"
    )
)

residual_df_dst = (
    residual_df.copy()
)

residual_df_dst[
    "local_hour"
] = (
    local_index.hour
)

residual_df_dst[
    "local_minute"
] = (
    local_index.minute
)

residual_df_dst[
    "local_dow"
] = (
    local_index.dayofweek
)

residual_df_dst[
    "local_month"
] = (
    local_index.month
)

In [22]:
local_hour_errors = (
    grouped_error_table(
        residual_df_dst,
        "local_hour"
    )
)

print(
    local_hour_errors
    .sort_values(
        "rmse",
        ascending=False
    )
    .head(10)
)

              n       bias         mae        rmse
local_hour                                        
13          730  16.216824  155.099094  233.666364
14          730 -15.251054  154.633598  212.952315
19          730 -18.867876  152.276298  208.239205
18          730 -18.468857  153.673905  197.559745
12          730   8.858325  138.416148  196.728805
20          730  -5.359457  139.967465  193.553392
1           730  -9.694270  128.642399  189.050074
7           730   0.150565  136.539449  181.005277
17          730  -5.029989  140.909783  180.953752
10          730  -1.529922  138.423506  179.422984


In [23]:
dst_mask_2025 = (
    dst_df_2025[
        "dst_window"
    ]
)

tmp = residual_df.copy()

tmp["dst_window"] = (
    dst_mask_2025
    .reindex(
        tmp.index
    )
    .fillna(False)
)


# Dimanches
sunday_all = (
    tmp.index.dayofweek
    == 6
)

sunday_normal = (
    sunday_all
    &
    ~tmp["dst_window"]
)


rmse_sunday_all = np.sqrt(
    np.mean(
        tmp.loc[
            sunday_all,
            "residual"
        ]**2
    )
)

rmse_sunday_normal = np.sqrt(
    np.mean(
        tmp.loc[
            sunday_normal,
            "residual"
        ]**2
    )
)


print(
    "Sunday RMSE all :",
    rmse_sunday_all
)

print(
    "Sunday RMSE excluding DST :",
    rmse_sunday_normal
)

Sunday RMSE all : 225.67517017233013
Sunday RMSE excluding DST : 157.2436678083444


In [24]:
october = (
    tmp.index.month
    == 10
)

october_normal = (
    october
    &
    ~tmp["dst_window"]
)


rmse_oct_all = np.sqrt(
    np.mean(
        tmp.loc[
            october,
            "residual"
        ]**2
    )
)

rmse_oct_normal = np.sqrt(
    np.mean(
        tmp.loc[
            october_normal,
            "residual"
        ]**2
    )
)


print(
    "October RMSE all :",
    rmse_oct_all
)

print(
    "October RMSE excluding DST :",
    rmse_oct_normal
)

October RMSE all : 241.42424058170533
October RMSE excluding DST : 182.74196581666772


### Partie C — Analyse

Le diagnostic des résidus met en évidence une forte hétérogénéité des erreurs.

Le modèle présente un biais global faible :

$$
-3.26\text{ MW}
$$

mais certaines périodes produisent des erreurs très importantes.

Les erreurs résiduelles présentent très peu d'autocorrélation aux lags classiques :

$$
\rho_{30min}=0.078
$$

$$
\rho_{1h}=-0.005
$$

$$
\rho_{1jour}=0.036
$$

$$
\rho_{1semaine}=0.006
$$

Il reste donc peu de structure temporelle linéaire classique à exploiter.

En revanche, les plus fortes erreurs sont fortement concentrées autour du 30 mars 2025 et du 26 octobre 2025.

Ces dates correspondent aux deux changements d'heure européens de 2025.

Cela suggère l'existence d'un régime particulier lié à la différence entre le temps UTC utilisé par les données et le temps local qui structure l'activité et la consommation électrique.

La prochaine étape consiste donc à mesurer quantitativement l'importance de ce régime avant de construire un correcteur spécialisé.

## Étape 2b — Calendar & Special Event Audit

### Partie A — Mise en pratique

In [25]:
from dateutil.easter import easter


def french_holidays(year):

    easter_day = pd.Timestamp(
        easter(year)
    )

    holidays_dict = {

        pd.Timestamp(f"{year}-01-01"):
            "New Year",

        easter_day + pd.Timedelta(days=1):
            "Easter Monday",

        pd.Timestamp(f"{year}-05-01"):
            "Labour Day",

        pd.Timestamp(f"{year}-05-08"):
            "Victory 1945",

        easter_day + pd.Timedelta(days=39):
            "Ascension",

        easter_day + pd.Timedelta(days=50):
            "Whit Monday",

        pd.Timestamp(f"{year}-07-14"):
            "Bastille Day",

        pd.Timestamp(f"{year}-08-15"):
            "Assumption",

        pd.Timestamp(f"{year}-11-01"):
            "All Saints",

        pd.Timestamp(f"{year}-11-11"):
            "Armistice",

        pd.Timestamp(f"{year}-12-25"):
            "Christmas"
    }

    return holidays_dict

In [26]:
holiday_dict = {}

for year in range(
    2013,
    2027
):

    holiday_dict.update(
        french_holidays(year)
    )

print(
    len(holiday_dict)
)

154


In [27]:
local_index = (
    residual_df.index
    .tz_convert(
        "Europe/Paris"
    )
)

local_dates = pd.DatetimeIndex(
    local_index.date
)

In [28]:
event_df = residual_df.copy()

event_df["local_date"] = (
    local_dates
)

event_df["holiday_name"] = [
    holiday_dict.get(
        pd.Timestamp(d),
        "Normal"
    )
    for d in local_dates
]

event_df["is_holiday"] = (
    event_df["holiday_name"]
    != "Normal"
)

In [29]:
holiday_dates = set(
    holiday_dict.keys()
)


event_df["holiday_eve"] = [
    (
        pd.Timestamp(d)
        + pd.Timedelta(days=1)
    )
    in holiday_dates

    for d in local_dates
]


event_df["after_holiday"] = [
    (
        pd.Timestamp(d)
        - pd.Timedelta(days=1)
    )
    in holiday_dates

    for d in local_dates
]

In [30]:
def is_bridge_day(date):

    date = pd.Timestamp(date)

    weekday = date.dayofweek

    # lundi avant mardi férié
    if weekday == 0:

        return (
            date
            + pd.Timedelta(days=1)
        ) in holiday_dates

    # vendredi après jeudi férié
    if weekday == 4:

        return (
            date
            - pd.Timedelta(days=1)
        ) in holiday_dates

    return False

In [31]:
event_df["is_bridge"] = [
    is_bridge_day(d)
    for d in local_dates
]

In [32]:
event_df["christmas_period"] = [
    (
        (d.month == 12 and d.day >= 23)
        or
        (d.month == 1 and d.day <= 2)
    )

    for d in local_index
]

In [33]:
def event_metrics(
    df,
    mask,
    name
):

    r = df.loc[
        mask,
        "residual"
    ].to_numpy()

    return {
        "Event":
            name,

        "n":
            len(r),

        "Bias":
            np.mean(r),

        "MAE":
            np.mean(
                np.abs(r)
            ),

        "RMSE":
            np.sqrt(
                np.mean(
                    r**2
                )
            ),

        "SSE":
            np.sum(
                r**2
            )
    }

In [34]:
event_results = []

event_results.append(
    event_metrics(
        event_df,
        event_df["is_holiday"],
        "Holiday"
    )
)

event_results.append(
    event_metrics(
        event_df,
        event_df["holiday_eve"],
        "Holiday eve"
    )
)

event_results.append(
    event_metrics(
        event_df,
        event_df["after_holiday"],
        "After holiday"
    )
)

event_results.append(
    event_metrics(
        event_df,
        event_df["is_bridge"],
        "Bridge day"
    )
)

event_results.append(
    event_metrics(
        event_df,
        event_df["christmas_period"],
        "Christmas/New Year"
    )
)

event_results = pd.DataFrame(
    event_results
)

In [35]:
total_sse = (
    event_df["residual"] ** 2
).sum()

event_results["SSE_share_%"] = (
    100
    * event_results["SSE"]
    / total_sse
)

print(
    event_results
    .sort_values(
        "RMSE",
        ascending=False
    )
)

                Event    n       Bias         MAE        RMSE           SSE  \
0             Holiday  526 -52.240196  169.621249  224.430037  2.649401e+07   
4  Christmas/New Year  525 -29.162292  164.810137  218.214447  2.499921e+07   
1         Holiday eve  527 -12.596909  141.539904  185.715926  1.817644e+07   
2       After holiday  528   8.596575  136.666781  176.900771  1.652317e+07   
3          Bridge day  240 -16.324345  132.787690  167.348897  6.721357e+06   

   SSE_share_%  
0     4.796096  
4     4.525499  
1     3.290403  
2     2.991118  
3     1.216738  


In [36]:
holiday_detail = (
    event_df[
        event_df["is_holiday"]
    ]
    .groupby(
        "holiday_name"
    )
    .agg(
        n=("residual", "size"),

        bias=(
            "residual",
            "mean"
        ),

        mae=(
            "abs_error",
            "mean"
        ),

        mse=(
            "sq_error",
            "mean"
        )
    )
)

holiday_detail["rmse"] = np.sqrt(
    holiday_detail["mse"]
)

holiday_detail = (
    holiday_detail
    .drop(
        columns="mse"
    )
    .sort_values(
        "rmse",
        ascending=False
    )
)

print(
    holiday_detail
)

                n        bias         mae        rmse
holiday_name                                         
Christmas      48  -91.334442  258.641427  328.238165
New Year       46 -146.308221  194.866781  264.557495
Armistice      48  -32.832220  188.952098  256.941094
All Saints     48   18.776962  173.251716  225.942567
Labour Day     48  -50.674696  160.863368  213.854086
Ascension      48  -46.075490  174.673099  212.927139
Easter Monday  48  -62.019077  158.424567  206.634843
Victory 1945   48  -64.836558  157.536103  200.970133
Assumption     48  -33.163732  142.038353  180.118730
Bastille Day   48  -38.821525  136.312119  175.803266
Whit Monday    48  -31.272655  121.326003  150.223393


### Conclusion — Calendar & Special Events Audit

Les erreurs restantes présentent une forte dépendance aux événements calendaires.

Les jours fériés ont un RMSE de :

$$
224.43\text{ MW}
$$

contre :

$$
177.58\text{ MW}
$$

sur l'ensemble de 2025.

Les difficultés sont cependant très hétérogènes selon l'événement.

Les RMSE les plus élevés sont notamment observés pour :

$$
Christmas = 328.24\text{ MW}
$$

$$
New\ Year = 264.56\text{ MW}
$$

$$
Armistice = 256.94\text{ MW}
$$

À l'inverse, certains jours spéciaux comme Whit Monday ne sont pas
particulièrement difficiles.

Les événements calendaires doivent donc être modélisés de manière
plus fine qu'avec une unique variable binaire `is_holiday`.

Combiné au très fort effet précédemment observé pour les changements
d'heure DST, cela suggère que les principales erreurs restantes
proviennent davantage de régimes calendaires rares que d'une
dépendance temporelle classique non capturée.

## Étape 3 — Correction SARIMAX des résidus

### Partie A — Mise en pratique
### A.1 — Construire les variables exogènes

In [37]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge


def build_calendar_exog(index):

    index = pd.DatetimeIndex(index)

    local = index.tz_convert(
        "Europe/Paris"
    )

    dates = pd.DatetimeIndex(
        local.date
    )

    df = pd.DataFrame(
        index=index
    )

    # =========================
    # Heure locale
    # =========================

    hour = (
        local.hour
        + local.minute / 60
    )

    df["hour_sin"] = np.sin(
        2 * np.pi * hour / 24
    )

    df["hour_cos"] = np.cos(
        2 * np.pi * hour / 24
    )

    # =========================
    # Semaine
    # =========================

    dow = local.dayofweek

    df["dow_sin"] = np.sin(
        2 * np.pi * dow / 7
    )

    df["dow_cos"] = np.cos(
        2 * np.pi * dow / 7
    )

    # =========================
    # Jours fériés
    # =========================

    df["is_holiday"] = [
        int(
            pd.Timestamp(d)
            in holiday_dates
        )
        for d in dates
    ]

    df["holiday_eve"] = [
        int(
            pd.Timestamp(d)
            + pd.Timedelta(days=1)
            in holiday_dates
        )
        for d in dates
    ]

    df["after_holiday"] = [
        int(
            pd.Timestamp(d)
            - pd.Timedelta(days=1)
            in holiday_dates
        )
        for d in dates
    ]

    # =========================
    # Noël / Nouvel An
    # =========================

    df["year_end_period"] = [
        int(
            (
                d.month == 12
                and d.day >= 23
            )
            or
            (
                d.month == 1
                and d.day <= 2
            )
        )
        for d in local
    ]

    # =========================
    # DST = changement d'heure
    # fenêtre ±24 h
    # =========================

    dst_flag = np.zeros(
        len(index),
        dtype=int
    )

    all_dst = (
        dst_2024
        + dst_2025
    )

    for transition in all_dst:

        distance = np.abs(
            (
                index
                - transition
            )
            / pd.Timedelta(hours=1)
        )

        dst_flag[
            distance <= 24
        ] = 1

    df["dst_window_24h"] = (
        dst_flag
    )

    return df.astype(float)

In [38]:
exog_2024 = build_calendar_exog(
    idx_2024
)

exog_2025 = build_calendar_exog(
    idx_2025
)

print(exog_2024.shape)
print(exog_2025.shape)

print(exog_2024.columns)

(4416, 9)
(17517, 9)
Index(['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_holiday',
       'holiday_eve', 'after_holiday', 'year_end_period', 'dst_window_24h'],
      dtype='str')


In [2]:
# ============================================================
# FONCTION D'ÉVALUATION DES MODÈLES DE RÉGRESSION
# ============================================================

import numpy as np

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


def regression_metrics(
    y_true,
    y_pred
):

    y_true = np.asarray(
        y_true
    )

    y_pred = np.asarray(
        y_pred
    )

    # MSE = Mean Squared Error
    mse = mean_squared_error(
        y_true,
        y_pred
    )

    # RMSE = Root Mean Squared Error
    rmse = np.sqrt(
        mse
    )

    # MAE = Mean Absolute Error
    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    nrmse_pct = (
    100
    * np.sqrt(mse)
    / np.mean(y)
)

    # MAPE = Mean Absolute Percentage Error
    # protection contre une éventuelle division par zéro
    mask = np.abs(y_true) > 1e-8

    mape = (
        np.mean(
            np.abs(
                (
                    y_true[mask]
                    - y_pred[mask]
                )
                /
                y_true[mask]
            )
        )
        * 100
    )

    # R² = coefficient de détermination
    r2 = r2_score(
        y_true,
        y_pred
    )

    return {
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE_%": mape,
        "NRMSE_%": nrmse_pct,
        "R2": r2
    }


print(
    "regression_metrics prête."
)

regression_metrics prête.


In [40]:
# ============================================================
# RECONSTRUCTION CALENDAR RIDGE — SANS SARIMAX
# ============================================================

import numpy as np
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge


# ------------------------------------------------------------
# 1. Vérifications
# ------------------------------------------------------------

print("Exog 2024 :", exog_2024.shape)
print("Exog 2025 :", exog_2025.shape)

assert len(exog_2024) == len(residual_2024)
assert len(exog_2025) == len(pred_2025)
assert len(pred_2025) == len(y_2025)


# ------------------------------------------------------------
# 2. Standardisation
# Apprentissage uniquement sur 2024
# ------------------------------------------------------------

scaler_event = StandardScaler()

X_event_2024 = scaler_event.fit_transform(
    exog_2024
)

X_event_2025 = scaler_event.transform(
    exog_2025
)


# ------------------------------------------------------------
# 3. Calendar Ridge
# ------------------------------------------------------------

event_ridge = Ridge(
    alpha=10.0
)

event_ridge.fit(
    X_event_2024,
    residual_2024
)


# ------------------------------------------------------------
# 4. Correction résiduelle 2025
# ------------------------------------------------------------

event_correction_2025 = (
    event_ridge.predict(
        X_event_2025
    )
)


# ------------------------------------------------------------
# 5. Prévision finale
# ------------------------------------------------------------

pred_event_ridge_2025 = (
    np.asarray(pred_2025)
    +
    event_correction_2025
)


# ------------------------------------------------------------
# 6. Métriques
# ------------------------------------------------------------

event_ridge_metrics = regression_metrics(
    y_2025,
    pred_event_ridge_2025
)

patch_metrics_current = regression_metrics(
    y_2025,
    pred_2025
)


print(
    "Patch baseline :",
    patch_metrics_current
)

print(
    "\nPatch + Calendar Ridge :",
    event_ridge_metrics
)


# ------------------------------------------------------------
# 7. Gain
# ------------------------------------------------------------

gain_event_ridge = (
    100
    * (
        patch_metrics_current["RMSE"]
        -
        event_ridge_metrics["RMSE"]
    )
    /
    patch_metrics_current["RMSE"]
)

print(
    "\nGain RMSE :",
    gain_event_ridge,
    "%"
)


# ------------------------------------------------------------
# 8. Sauvegarde
# ------------------------------------------------------------

joblib.dump(
    {
        "model":
            event_ridge,

        "scaler":
            scaler_event,

        "prediction_2025":
            pred_event_ridge_2025,

        "correction_2025":
            event_correction_2025,

        "metrics":
            event_ridge_metrics,

        "gain_rmse_pct":
            gain_event_ridge,

        "exog_columns":
            list(exog_2024.columns)
    },

    "models/block7_calendar_ridge_state.joblib"
)

print(
    "\nCalendar Ridge reconstruit et sauvegardé."
)

Exog 2024 : (4416, 9)
Exog 2025 : (17517, 9)
Patch baseline : {'MSE': 31535.528930303284, 'RMSE': np.float64(177.58245670759058), 'MAE': 128.24532895554032, 'MAPE_%': np.float64(0.2586217469956298), 'R2': 0.9997203651845527}

Patch + Calendar Ridge : {'MSE': 31561.68550018652, 'RMSE': np.float64(177.65608770933383), 'MAE': 128.55532088710905, 'MAPE_%': np.float64(0.2591845718545704), 'R2': 0.99972013324655}

Gain RMSE : -0.04146299308410472 %

Calendar Ridge reconstruit et sauvegardé.


In [41]:
event_ridge_metrics = regression_metrics(
    y_2025,
    pred_event_ridge_2025
)

print(
    "Patch baseline :",
    state["patch_metrics_2025"]
)

print(
    "\nPatch + Calendar Ridge :",
    event_ridge_metrics
)

Patch baseline : {'MSE': 31535.528930303284, 'RMSE': np.float64(177.58245670759058), 'MAE': 128.24532895554032, 'MAPE_%': 0.2586217469956298, 'R2': 0.9997203651845527}

Patch + Calendar Ridge : {'MSE': 31561.68550018652, 'RMSE': np.float64(177.65608770933383), 'MAE': 128.55532088710905, 'MAPE_%': np.float64(0.2591845718545704), 'R2': 0.99972013324655}


In [42]:
gain_event_ridge = (
    100
    * (
        state["patch_metrics_2025"]["RMSE"]
        - event_ridge_metrics["RMSE"]
    )
    / state["patch_metrics_2025"]["RMSE"]
)

print(
    "Gain RMSE :",
    gain_event_ridge,
    "%"
)

Gain RMSE : -0.04146299308410472 %


In [43]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
import time

In [44]:
start = time.time()

sarimax_model = SARIMAX(
    residual_2024,

    exog=exog_2024,

    order=(
        1, 0, 1
    ),

    seasonal_order=(
        1, 0, 0, 48
    ),

    trend="c",

    enforce_stationarity=False,
    enforce_invertibility=False
)

C:\Users\thiba\AppData\Roaming\Python\Python312\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency 30min will be used.
  self._init_dates(dates, freq)
C:\Users\thiba\AppData\Roaming\Python\Python312\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency 30min will be used.
  self._init_dates(dates, freq)


In [45]:
sarimax_result = sarimax_model.fit(
    disp=False,
    maxiter=30
)

elapsed = (
    time.time()
    - start
)

print(
    "Temps :",
    elapsed,
    "secondes"
)

Temps : 64.50025153160095 secondes


In [46]:
sarimax_residual_2025 = (
    sarimax_result.forecast(
        steps=len(idx_2025),
        exog=exog_2025
    )
)

In [47]:
pred_sarimax_2025 = (
    pred_2025
    + np.asarray(
        sarimax_residual_2025
    )
)

In [48]:
sarimax_metrics = regression_metrics(
    y_2025,
    pred_sarimax_2025
)

print(
    "Patch :",
    state["patch_metrics_2025"]
)

print(
    "\nPatch + Calendar Ridge :",
    event_ridge_metrics
)

print(
    "\nPatch + SARIMAX :",
    sarimax_metrics
)

Patch : {'MSE': 31535.528930303284, 'RMSE': np.float64(177.58245670759058), 'MAE': 128.24532895554032, 'MAPE_%': 0.2586217469956298, 'R2': 0.9997203651845527}

Patch + Calendar Ridge : {'MSE': 31561.68550018652, 'RMSE': np.float64(177.65608770933383), 'MAE': 128.55532088710905, 'MAPE_%': np.float64(0.2591845718545704), 'R2': 0.99972013324655}

Patch + SARIMAX : {'MSE': 31558.09073217025, 'RMSE': np.float64(177.64597021089517), 'MAE': 128.557775999611, 'MAPE_%': np.float64(0.2591815657715437), 'R2': 0.9997201651224159}


In [49]:
sarimax_gain = (
    100
    * (
        state["patch_metrics_2025"]["RMSE"]
        - sarimax_metrics["RMSE"]
    )
    / state["patch_metrics_2025"]["RMSE"]
)

print(
    "\nSARIMAX gain :",
    sarimax_gain,
    "%"
)


SARIMAX gain : -0.03576564063936415 %


In [50]:
joblib.dump(
    {
        "calendar_ridge_metrics":
            event_ridge_metrics,

        "sarimax_metrics":
            sarimax_metrics,

        "sarimax_gain":
            sarimax_gain,

        "calendar_ridge_gain":
            gain_event_ridge,

        "sarimax_prediction":
            pred_sarimax_2025,

        "calendar_ridge_prediction":
            pred_event_ridge_2025,

        "exog_columns":
            list(exog_2024.columns)
    },
    "models/block7_calendar_sarimax_results.joblib"
)

['models/block7_calendar_sarimax_results.joblib']

### Conclusion — Calendar Ridge et SARIMAX

L'ajout global de variables calendaires ne permet pas d'améliorer
le Patch Transformer.

Le RMSE passe de :

$$
177.5825
$$

à :

$$
177.6561
$$

avec Calendar Ridge et :

$$
177.6460
$$

avec SARIMAX.

Les deux modèles dégradent donc légèrement la performance globale.

Ce résultat ne signifie pas que les événements calendaires sont
inutiles. L'analyse précédente a montré au contraire que DST et
certains jours fériés présentent des erreurs très supérieures à la
normale.

Le problème semble plutôt provenir d'une correction appliquée à
l'ensemble des observations alors que ces régimes sont rares.

La suite consiste donc à construire un expert spécialisé qui ne
modifie la prédiction que pendant les régimes calendaires difficiles.

## Étape 4 — Specialized Event Expert

In [51]:
stage1_state = joblib.load(
    "models/block6_stage1_full_2024_2025.joblib"
)

idx_full_2024 = stage1_state[
    "index_2024"
]

r2_full_2024 = stage1_state[
    "r2_2024"
]

pred_stage1_full_2024 = stage1_state[
    "pred_stage1_2024"
]

idx_full_2025 = stage1_state[
    "index_2025"
]

pred_stage1_full_2025 = stage1_state[
    "pred_stage1_2025"
]

print(
    "2024 complet :",
    len(idx_full_2024)
)

print(
    "2025 :",
    len(idx_full_2025)
)

2024 complet : 17568
2025 : 17517


In [52]:
from sklearn.ensemble import (
    HistGradientBoostingRegressor
)

In [53]:
event_X_2024 = build_calendar_exog(
    idx_full_2024
)

event_X_2025 = build_calendar_exog(
    idx_full_2025
)

In [54]:
cut = pd.Timestamp(
    "2024-10-01",
    tz=idx_full_2024.tz
)

expert_train_mask = (
    idx_full_2024 < cut
)

expert_val_mask = (
    idx_full_2024 >= cut
)

In [55]:
event_columns = [
    "is_holiday",
    "holiday_eve",
    "after_holiday",
    "year_end_period",
    "dst_window_24h"
]

event_mask_2024 = (
    event_X_2024[
        event_columns
    ]
    .max(axis=1)
    .astype(bool)
)

In [56]:
sample_weight = np.ones(
    len(event_X_2024)
)

sample_weight[
    event_mask_2024.to_numpy()
] = 8.0

In [57]:
event_expert = (
    HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_iter=150,
        max_depth=3,
        l2_regularization=10.0,
        early_stopping=True,
        random_state=42
    )
)

event_expert.fit(
    event_X_2024.loc[
        expert_train_mask
    ],

    r2_full_2024[
        expert_train_mask
    ],

    sample_weight=sample_weight[
        expert_train_mask
    ]
)

,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.05
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees.",150
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",3
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",10.0
,"early_stopping early_stopping: 'auto' or bool, default='auto'If 'auto', early stopping is enabled if the sample size is larger than10000 or if `X_val` and `y_val` are passed to `fit`. If True, early stoppingis enabled, otherwise early stopping is disabled... versionadded:: 0.23",True
,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'squared_error', 'absolute_error', 'gamma', 'poisson', 'quantile'}, default='squared_error'The loss function to use in the boosting process. Note that the""squared error"", ""gamma"" and ""poisson"" losses actually implement""half least squares loss"", ""half gamma deviance"" and ""half poissondeviance"" to simplify the computation of the gradient. Furthermore,""gamma"" and ""poisson"" losses internally use a log-link, ""gamma""requires ``y > 0`` and ""poisson"" requires ``y >= 0``.""quantile"" uses the pinball loss... versionchanged:: 0.23 Added option 'poisson'... versionchanged:: 1.1 Added option 'quantile'... versionchanged:: 1.3 Added option 'gamma'.",'squared_error'
,"quantile quantile: float, default=NoneIf loss is ""quantile"", this parameter specifies which quantile to be estimatedand must be between 0 and 1.",None
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0


In [58]:
patch_q4 = pd.Series(
    state["pred_patch_2024"],
    index=state["val_index_2024"]
)

In [59]:
stage1_2024_series = pd.Series(
    pred_stage1_full_2024,
    index=idx_full_2024
)

stage1_q4 = (
    stage1_2024_series.loc[
        patch_q4.index
    ]
    .to_numpy()
)

In [60]:
patch_correction_q4 = (
    patch_q4.to_numpy()
    - stage1_q4
)

In [61]:
expert_correction_q4 = (
    event_expert.predict(
        event_X_2024.loc[
            patch_q4.index
        ]
    )
)

In [62]:
q4_event_mask = (
    event_X_2024
    .loc[
        patch_q4.index,
        event_columns
    ]
    .max(axis=1)
    .astype(bool)
    .to_numpy()
)

In [63]:
lambda_results = []

y_q4 = (
    y.loc[
        patch_q4.index
    ]
    .to_numpy()
)

for lam in np.linspace(
    0,
    1,
    11
):

    correction = (
        patch_correction_q4.copy()
    )

    correction[
        q4_event_mask
    ] = (
        (1 - lam)
        * patch_correction_q4[
            q4_event_mask
        ]

        +

        lam
        * expert_correction_q4[
            q4_event_mask
        ]
    )

    pred = (
        stage1_q4
        + correction
    )

    metrics = regression_metrics(
        y_q4,
        pred
    )

    lambda_results.append(
        {
            "lambda":
                lam,

            "RMSE":
                metrics["RMSE"]
        }
    )


lambda_df = pd.DataFrame(
    lambda_results
)

print(
    lambda_df
)

    lambda        RMSE
0      0.0  201.315456
1      0.1  201.129741
2      0.2  200.987483
3      0.3  200.888776
4      0.4  200.833683
5      0.5  200.822240
6      0.6  200.854456
7      0.7  200.930308
8      0.8  201.049748
9      0.9  201.212697
10     1.0  201.419051


In [64]:
best_lambda = (
    lambda_df.loc[
        lambda_df["RMSE"].idxmin(),
        "lambda"
    ]
)

print(
    "Best lambda :",
    best_lambda
)

Best lambda : 0.5


In [65]:
# ============================================================
# ÉVALUATION 2025 — PATCH + EVENT EXPERT
# Lambda choisie uniquement sur Q4 2024
# ============================================================

best_lambda = 0.5


# ------------------------------------------------------------
# 1. Séries Stage 1 et Patch alignées
# ------------------------------------------------------------

stage1_2025_series = pd.Series(
    pred_stage1_full_2025,
    index=idx_full_2025,
    name="Stage1"
)

patch_2025_series = pd.Series(
    pred_2025,
    index=idx_2025,
    name="Patch"
)


# ------------------------------------------------------------
# 2. Index commun
# ------------------------------------------------------------

common_2025 = (
    patch_2025_series.index
    .intersection(
        stage1_2025_series.index
    )
)

print(
    "Observations communes :",
    len(common_2025)
)

Observations communes : 17517


In [66]:
# ============================================================
# CORRECTION PATCH
# ============================================================

stage1_common = (
    stage1_2025_series
    .loc[common_2025]
    .to_numpy()
)

patch_common = (
    patch_2025_series
    .loc[common_2025]
    .to_numpy()
)

patch_correction_2025 = (
    patch_common
    - stage1_common
)

In [67]:
# ============================================================
# CORRECTION DE L'EXPERT
# ============================================================

expert_correction_2025 = (
    event_expert.predict(
        event_X_2025.loc[
            common_2025
        ]
    )
)

In [68]:
# ============================================================
# EVENT GATE
# ============================================================

event_mask_2025 = (
    event_X_2025
    .loc[
        common_2025,
        event_columns
    ]
    .max(axis=1)
    .astype(bool)
    .to_numpy()
)

print(
    "Observations événement :",
    event_mask_2025.sum()
)

print(
    "Part événement :",
    100
    * event_mask_2025.mean(),
    "%"
)

Observations événement : 2015
Part événement : 11.50311126334418 %


In [69]:
# ============================================================
# MIXTURE PATCH + EXPERT
# ============================================================

final_correction_2025 = (
    patch_correction_2025.copy()
)

final_correction_2025[
    event_mask_2025
] = (

    (1 - best_lambda)
    * patch_correction_2025[
        event_mask_2025
    ]

    +

    best_lambda
    * expert_correction_2025[
        event_mask_2025
    ]
)

In [70]:
pred_event_expert_2025 = (
    stage1_common
    + final_correction_2025
)

In [71]:
y_common_2025 = (
    y.loc[
        common_2025
    ]
    .to_numpy()
)

In [72]:
expert_metrics_2025 = regression_metrics(
    y_common_2025,
    pred_event_expert_2025
)

patch_common_metrics = regression_metrics(
    y_common_2025,
    patch_common
)

print(
    "Patch :",
    patch_common_metrics
)

print(
    "\nPatch + Event Expert :",
    expert_metrics_2025
)

Patch : {'MSE': 31535.528930303284, 'RMSE': np.float64(177.58245670759058), 'MAE': 128.24532895554032, 'MAPE_%': np.float64(0.2586217469956298), 'R2': 0.9997203651845527}

Patch + Event Expert : {'MSE': 31316.22185500486, 'RMSE': np.float64(176.9638998637995), 'MAE': 127.84698411691858, 'MAPE_%': np.float64(0.2576024858824899), 'R2': 0.9997223098449279}


In [73]:
expert_gain = (
    100
    * (
        patch_common_metrics["RMSE"]
        - expert_metrics_2025["RMSE"]
    )
    / patch_common_metrics["RMSE"]
)

print(
    "\nGain RMSE 2025 :",
    expert_gain,
    "%"
)


Gain RMSE 2025 : 0.3483209182141265 %


In [74]:
patch_event_metrics = regression_metrics(
    y_common_2025[
        event_mask_2025
    ],

    patch_common[
        event_mask_2025
    ]
)

expert_event_metrics = regression_metrics(
    y_common_2025[
        event_mask_2025
    ],

    pred_event_expert_2025[
        event_mask_2025
    ]
)

print(
    "Patch — événements :",
    patch_event_metrics
)

print(
    "\nPatch + Expert — événements :",
    expert_event_metrics
)

Patch — événements : {'MSE': 72040.59873265093, 'RMSE': np.float64(268.4037979102586), 'MAE': 170.03209674750002, 'MAPE_%': np.float64(0.3565883375149212), 'R2': 0.9993504157153043}

Patch + Expert — événements : {'MSE': 70134.09648054042, 'RMSE': np.float64(264.8284283843795), 'MAE': 166.569165463065, 'MAPE_%': np.float64(0.34772759462714353), 'R2': 0.9993676064927755}


In [75]:
event_gain = (
    100
    * (
        patch_event_metrics["RMSE"]
        - expert_event_metrics["RMSE"]
    )
    / patch_event_metrics["RMSE"]
)

print(
    "\nGain sur événements :",
    event_gain,
    "%"
)


Gain sur événements : 1.3320860411500428 %


In [76]:
joblib.dump(
    {
        "lambda":
            best_lambda,

        "validation_lambda_table":
            lambda_df,

        "patch_metrics_2025":
            patch_common_metrics,

        "expert_metrics_2025":
            expert_metrics_2025,

        "global_gain_pct":
            expert_gain,

        "patch_event_metrics":
            patch_event_metrics,

        "expert_event_metrics":
            expert_event_metrics,

        "event_gain_pct":
            event_gain,

        "prediction_2025":
            pred_event_expert_2025,

        "index_2025":
            common_2025,

        "event_mask_2025":
            event_mask_2025
    },

    "models/block7_event_expert_results.joblib"
)

print(
    "Event Expert sauvegardé."
)

Event Expert sauvegardé.


### Conclusion — Specialized Event Expert

L'expert spécialisé sur les régimes calendaires améliore la
généralisation du Patch Transformer.

Le coefficient de mélange optimal, sélectionné sur Q4 2024, est :

$$
\lambda = 0.5
$$

Sur 2025, le RMSE global passe de :

$$
177.58\text{ MW}
$$

à :

$$
176.96\text{ MW}
$$

soit une amélioration d'environ :

$$
0.35\%
$$

Sur les périodes événementielles uniquement, le RMSE passe de :

$$
268.40\text{ MW}
$$

à :

$$
264.83\text{ MW}
$$

soit une amélioration d'environ :

$$
1.33\%
$$

Contrairement à Calendar Ridge et SARIMAX, l'expert spécialisé
n'altère pas les périodes normales.

Cela confirme que les informations calendaires sont principalement
utiles comme mécanisme de sélection de régime plutôt que comme
correction globale.

## Étape 5 — Event-Specific MoE Audit

In [77]:
# ============================================================
# ÉTAPE 5 — PERFORMANCE PAR TYPE D'ÉVÉNEMENT
# ============================================================

event_analysis = []

for event_name in event_columns:

    mask = (
        event_X_2025
        .loc[
            common_2025,
            event_name
        ]
        .astype(bool)
        .to_numpy()
    )

    if mask.sum() == 0:
        continue

    patch_m = regression_metrics(
        y_common_2025[mask],
        patch_common[mask]
    )

    expert_m = regression_metrics(
        y_common_2025[mask],
        pred_event_expert_2025[mask]
    )

    gain_rmse = (
        100
        * (
            patch_m["RMSE"]
            - expert_m["RMSE"]
        )
        / patch_m["RMSE"]
    )

    gain_mae = (
        100
        * (
            patch_m["MAE"]
            - expert_m["MAE"]
        )
        / patch_m["MAE"]
    )

    event_analysis.append(
        {
            "Event":
                event_name,

            "n":
                mask.sum(),

            "Patch_RMSE":
                patch_m["RMSE"],

            "Expert_RMSE":
                expert_m["RMSE"],

            "RMSE_gain_%":
                gain_rmse,

            "Patch_MAE":
                patch_m["MAE"],

            "Expert_MAE":
                expert_m["MAE"],

            "MAE_gain_%":
                gain_mae
        }
    )


event_analysis_df = pd.DataFrame(
    event_analysis
)

print(
    event_analysis_df
    .sort_values(
        "RMSE_gain_%",
        ascending=False
    )
    .round(4)
)

             Event    n  Patch_RMSE  Expert_RMSE  RMSE_gain_%  Patch_MAE  \
0       is_holiday  526    224.4300     208.1030       7.2749   169.6212   
3  year_end_period  525    218.2144     212.9488       2.4131   164.8101   
2    after_holiday  528    176.9008     175.8302       0.6052   136.6668   
4   dst_window_24h  194    630.0140     630.8616      -0.1345   383.1735   
1      holiday_eve  527    185.7159     186.3377      -0.3348   141.5399   

   Expert_MAE  MAE_gain_%  
0    158.0032      6.8494  
3    162.2822      1.5338  
2    135.1023      1.1448  
4    383.1355      0.0099  
1    142.3021     -0.5385  


In [78]:
normal_mask = (
    ~event_mask_2025
)

normal_metrics = regression_metrics(
    y_common_2025[
        normal_mask
    ],
    patch_common[
        normal_mask
    ]
)

print(
    "Normal observations :",
    normal_mask.sum()
)

print(
    "Patch normal :",
    normal_metrics
)

Normal observations : 15502
Patch normal : {'MSE': 26270.5492082203, 'RMSE': np.float64(162.08192128741658), 'MAE': 122.81374999148412, 'MAPE_%': np.float64(0.24588773332665986), 'R2': 0.9997669239469029}


In [79]:
difference_normal = np.max(
    np.abs(
        patch_common[
            normal_mask
        ]
        -
        pred_event_expert_2025[
            normal_mask
        ]
    )
)

print(
    "Maximum difference outside events :",
    difference_normal
)

Maximum difference outside events : 0.0


In [80]:
joblib.dump(
    event_analysis_df,
    "models/block7_event_specific_analysis.joblib"
)

joblib.dump(
    event_expert,
    "models/block7_event_expert_model.joblib"
)

joblib.dump(
    {
        "lambda": best_lambda,
        "event_columns": event_columns,
        "rmse_2025": expert_metrics_2025["RMSE"],
        "baseline_rmse_2025": patch_common_metrics["RMSE"]
    },
    "models/block7_event_expert_config.joblib"
)

print(
    "Expert et configuration sauvegardés."
)

Expert et configuration sauvegardés.


In [81]:
# ============================================================
# GATE RAFFINE
# Expert uniquement sur les régimes où il apporte un gain
# ============================================================

refined_event_columns = [
    "is_holiday",
    "year_end_period",
    "after_holiday"
]

refined_event_mask_2025 = (
    event_X_2025
    .loc[
        common_2025,
        refined_event_columns
    ]
    .max(axis=1)
    .astype(bool)
    .to_numpy()
)

print(
    "Observations refined gate :",
    refined_event_mask_2025.sum()
)

print(
    "Part refined gate :",
    100 * refined_event_mask_2025.mean(),
    "%"
)

Observations refined gate : 1389
Part refined gate : 7.929439972598047 %


In [82]:
# ============================================================
# PATCH + EXPERT AVEC GATE RAFFINE
# ============================================================

pred_refined_2025 = (
    patch_common.copy()
)

# Correction finale uniquement sur le nouveau gate
refined_correction = (
    (1 - best_lambda)
    * patch_correction_2025[
        refined_event_mask_2025
    ]
    +
    best_lambda
    * expert_correction_2025[
        refined_event_mask_2025
    ]
)

pred_refined_2025[
    refined_event_mask_2025
] = (
    stage1_common[
        refined_event_mask_2025
    ]
    +
    refined_correction
)

In [83]:
refined_metrics = regression_metrics(
    y_common_2025,
    pred_refined_2025
)

print(
    "Patch :",
    patch_common_metrics
)

print(
    "\nAncien Event Expert :",
    expert_metrics_2025
)

print(
    "\nRefined Event Expert :",
    refined_metrics
)

Patch : {'MSE': 31535.528930303284, 'RMSE': np.float64(177.58245670759058), 'MAE': 128.24532895554032, 'MAPE_%': np.float64(0.2586217469956298), 'R2': 0.9997203651845527}

Ancien Event Expert : {'MSE': 31316.22185500486, 'RMSE': np.float64(176.9638998637995), 'MAE': 127.84698411691858, 'MAPE_%': np.float64(0.2576024858824899), 'R2': 0.9997223098449279}

Refined Event Expert : {'MSE': 31308.66995957857, 'RMSE': np.float64(176.94256118746154), 'MAE': 127.8516195648178, 'MAPE_%': np.float64(0.25761173413837657), 'R2': 0.9997223768098071}


In [84]:
refined_gain_vs_patch = (
    100
    * (
        patch_common_metrics["RMSE"]
        - refined_metrics["RMSE"]
    )
    / patch_common_metrics["RMSE"]
)

refined_gain_vs_old_expert = (
    100
    * (
        expert_metrics_2025["RMSE"]
        - refined_metrics["RMSE"]
    )
    / expert_metrics_2025["RMSE"]
)

print(
    "\nGain vs Patch :",
    refined_gain_vs_patch,
    "%"
)

print(
    "Gain vs ancien expert :",
    refined_gain_vs_old_expert,
    "%"
)


Gain vs Patch : 0.3603371256332513 %
Gain vs ancien expert : 0.012058208682327331 %


In [85]:
dst_mask = (
    event_X_2025
    .loc[
        common_2025,
        "dst_window_24h"
    ]
    .astype(bool)
    .to_numpy()
)

dst_refined_metrics = regression_metrics(
    y_common_2025[dst_mask],
    pred_refined_2025[dst_mask]
)

dst_patch_metrics = regression_metrics(
    y_common_2025[dst_mask],
    patch_common[dst_mask]
)

print(
    "DST Patch :",
    dst_patch_metrics
)

print(
    "\nDST refined system :",
    dst_refined_metrics
)

DST Patch : {'MSE': 396917.6790693806, 'RMSE': np.float64(630.0140308512031), 'MAE': 383.17351051886635, 'MAPE_%': np.float64(0.8195457689258876), 'R2': 0.9699822729126892}

DST refined system : {'MSE': 396917.6790693806, 'RMSE': np.float64(630.0140308512031), 'MAE': 383.17351051886635, 'MAPE_%': np.float64(0.8195457689258876), 'R2': 0.9699822729126892}


In [86]:
joblib.dump(
    {
        "event_columns":
            refined_event_columns,

        "lambda":
            best_lambda,

        "metrics":
            refined_metrics,

        "gain_vs_patch":
            refined_gain_vs_patch,

        "prediction_2025":
            pred_refined_2025,

        "event_mask_2025":
            refined_event_mask_2025,

        "index_2025":
            common_2025
    },

    "models/block7_refined_event_expert.joblib"
)

['models/block7_refined_event_expert.joblib']

### Conclusion — Event-Specific Expert Audit

L'analyse par type d'événement montre que l'expert n'est pas
uniformément utile.

Les gains les plus importants apparaissent pendant les jours fériés :

$$
RMSE : 224.43 \rightarrow 208.10\text{ MW}
$$

soit :

$$
7.27\%
$$

d'amélioration.

La période de Noël et du Nouvel An bénéficie également du modèle,
avec une amélioration d'environ :

$$
2.41\%
$$

En revanche, l'expert n'améliore pas les changements d'heure DST
et dégrade légèrement les veilles de jours fériés.

Le mécanisme de sélection est donc raffiné afin d'activer l'expert
principalement pour les régimes où son utilité a été observée.

En dehors des périodes événementielles, la différence avec le
Patch Transformer reste exactement nulle.

## Étape 6 — DST-Aware Local-Time Expert

### Partie A — Mise en pratique

In [87]:
def local_wall_lag(
    series,
    days
):

    series = (
        series
        .sort_index()
        .astype(float)
    )

    # ==========================================
    # Passage UTC -> heure locale française
    # ==========================================

    local_index = (
        series.index
        .tz_convert(
            "Europe/Paris"
        )
    )

    # Date locale
    local_date = pd.to_datetime(
        local_index.date
    )

    # Slot de demi-heure :
    #
    # 00:00 -> 0
    # 00:30 -> 1
    # ...
    # 23:30 -> 47

    slot = (
        local_index.hour * 2
        +
        local_index.minute // 30
    ).astype(int)


    # ==========================================
    # Table date / slot / valeur
    # ==========================================

    tmp = pd.DataFrame(
        {
            "date":
                local_date,

            "slot":
                slot,

            "value":
                series.to_numpy()
        }
    )


    # ==========================================
    # Une journée = 48 slots
    #
    # Automne :
    # heure répétée -> moyenne
    # ==========================================

    profile = (
        tmp
        .groupby(
            [
                "date",
                "slot"
            ]
        )["value"]
        .mean()
        .unstack("slot")
        .reindex(
            columns=range(48)
        )
    )


    # ==========================================
    # Printemps :
    # heure manquante -> interpolation
    # ==========================================

    profile = (
        profile
        .interpolate(
            axis=1,
            limit_direction="both"
        )
    )


    # ==========================================
    # Retour en format long
    #
    # IMPORTANT :
    # plus de dropna=False
    # avec les nouvelles versions pandas
    # ==========================================

    stacked = (
        profile
        .stack()
    )


    # ==========================================
    # Chercher le même horaire local
    # days jours auparavant
    # ==========================================

    target_date = (
        local_date
        -
        pd.to_timedelta(
            days,
            unit="D"
        )
    )


    target_key = (
        pd.MultiIndex.from_arrays(
            [
                target_date,
                slot
            ],
            names=[
                "date",
                "slot"
            ]
        )
    )


    result = (
        stacked
        .reindex(
            target_key
        )
        .to_numpy()
    )


    return pd.Series(
        result,
        index=series.index,
        dtype=float
    )

In [88]:
def build_dst_metadata(index):

    index = pd.DatetimeIndex(
        index
    )

    local = (
        index
        .tz_convert(
            "Europe/Paris"
        )
    )

    # Décalage UTC en heures
    offsets = np.array([
        ts.utcoffset().total_seconds()
        / 3600

        for ts in local
    ])

    delta_offset = np.diff(
        offsets
    )

    # Endroits où l'offset change
    change_positions = (
        np.where(
            delta_offset != 0
        )[0]
        + 1
    )

    transitions = (
        index[
            change_positions
        ]
    )

    directions = np.sign(
        delta_offset[
            change_positions - 1
        ]
    ).astype(int)

    # +1 = passage heure d'été
    # -1 = passage heure d'hiver

    hours_to_dst = np.full(
        len(index),
        np.inf
    )

    nearest_direction = np.zeros(
        len(index),
        dtype=int
    )

    for transition, direction in zip(
        transitions,
        directions
    ):

        distance = np.asarray(
            (
                index - transition
            )
            /
            pd.Timedelta(hours=1)
        )

        better = (
            np.abs(distance)
            <
            np.abs(hours_to_dst)
        )

        hours_to_dst[
            better
        ] = distance[
            better
        ]

        nearest_direction[
            better
        ] = direction

    result = pd.DataFrame(
        index=index
    )

    result[
        "hours_to_dst"
    ] = hours_to_dst

    result[
        "dst_spring"
    ] = (
        nearest_direction == 1
    ).astype(int)

    result[
        "dst_fall"
    ] = (
        nearest_direction == -1
    ).astype(int)

    result[
        "after_transition"
    ] = (
        hours_to_dst >= 0
    ).astype(int)

    return (
        result,
        transitions
    )

In [89]:
# ============================================================
# HISTORIQUE COMPLET
# ============================================================

common_history_index = (
    X.index
    .intersection(
        y.index
    )
)

y_history = (
    y.loc[
        common_history_index
    ]
    .astype(float)
)

X_history = (
    X.loc[
        common_history_index
    ]
)

In [90]:
print(
    "Construction lag local 1 jour..."
)

lag_local_1d = local_wall_lag(
    y_history,
    days=1
)

print(
    "Construction lag local 7 jours..."
)

lag_local_7d = local_wall_lag(
    y_history,
    days=7
)

Construction lag local 1 jour...
Construction lag local 7 jours...


In [91]:
lag_utc_1d = (
    y_history.shift(48)
)

lag_utc_7d = (
    y_history.shift(336)
)

In [92]:
dst_features = pd.DataFrame(
    index=common_history_index
)

dst_features[
    "lag_local_1d"
] = lag_local_1d

dst_features[
    "lag_local_7d"
] = lag_local_7d

dst_features[
    "lag_utc_1d"
] = lag_utc_1d

dst_features[
    "lag_utc_7d"
] = lag_utc_7d

In [93]:
dst_meta, dst_transitions = (
    build_dst_metadata(
        common_history_index
    )
)

dst_features = pd.concat(
    [
        dst_features,
        dst_meta
    ],
    axis=1
)

print(
    "Nombre de transitions détectées :",
    len(dst_transitions)
)

print(
    dst_transitions
)

Nombre de transitions détectées : 26
DatetimeIndex(['2013-03-31 01:00:00+00:00', '2013-10-27 01:00:00+00:00',
               '2014-03-30 01:00:00+00:00', '2014-10-26 01:00:00+00:00',
               '2015-03-29 01:00:00+00:00', '2015-10-25 01:00:00+00:00',
               '2016-03-27 01:00:00+00:00', '2016-10-30 01:00:00+00:00',
               '2017-03-26 01:00:00+00:00', '2017-10-29 01:00:00+00:00',
               '2018-03-25 01:00:00+00:00', '2018-10-28 01:00:00+00:00',
               '2019-03-31 01:00:00+00:00', '2019-10-27 01:00:00+00:00',
               '2020-03-29 01:00:00+00:00', '2020-10-25 01:00:00+00:00',
               '2021-03-28 01:00:00+00:00', '2021-10-31 01:00:00+00:00',
               '2022-03-27 01:00:00+00:00', '2022-10-30 01:00:00+00:00',
               '2023-03-26 01:00:00+00:00', '2023-10-29 01:00:00+00:00',
               '2024-03-31 01:00:00+00:00', '2024-10-27 01:00:00+00:00',
               '2025-03-30 01:00:00+00:00', '2025-10-26 01:00:00+00:00'],
             

In [94]:
dst_features[
    "hours_to_dst_scaled"
] = (
    dst_features[
        "hours_to_dst"
    ]
    .clip(
        -48,
        48
    )
    / 48
)

In [95]:
temp_candidates = [
    "temp_lag_1h",
    "temp_mean_24h",
    "temp_lag_24h"
]

temp_feature = next(
    (
        col
        for col in temp_candidates
        if col in X_history.columns
    ),
    None
)

print(
    "Température utilisée :",
    temp_feature
)

if temp_feature is not None:

    dst_features[
        "temperature"
    ] = (
        X_history[
            temp_feature
        ]
    )

Température utilisée : temp_lag_1h


In [96]:
dst_target = (
    y_history
    -
    dst_features[
        "lag_local_7d"
    ]
)

In [97]:
dst_meta_raw, _ = (
    build_dst_metadata(
        common_history_index
    )
)

dst_window_48 = (
    dst_meta_raw[
        "hours_to_dst"
    ].abs()
    <= 48
)

In [98]:
valid_dst = (
    dst_features
    .notna()
    .all(axis=1)
    &
    dst_target
    .notna()
)

In [99]:
from sklearn.ensemble import (
    HistGradientBoostingRegressor
)


train_dst_mask = (
    dst_window_48
    &
    valid_dst
    &
    (
        dst_features.index
        < "2024-01-01"
    )
)

print(
    "Observations DST train :",
    train_dst_mask.sum()
)

Observations DST train : 4246


In [100]:
dst_expert = (
    HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_iter=150,
        max_depth=3,
        min_samples_leaf=20,
        l2_regularization=10.0,
        early_stopping=True,
        random_state=42
    )
)

dst_expert.fit(
    dst_features.loc[
        train_dst_mask
    ],

    dst_target.loc[
        train_dst_mask
    ]
)

print(
    "DST expert entraîné."
)

DST expert entraîné.


In [101]:
patch_val_index = (
    state[
        "val_index_2024"
    ]
)

patch_val_prediction = np.asarray(
    state[
        "pred_patch_2024"
    ]
)

y_patch_val = np.asarray(
    state[
        "y_2024"
    ]
)

In [102]:
X_dst_val = (
    dst_features.loc[
        patch_val_index
    ]
)

dst_delta_val = (
    dst_expert.predict(
        X_dst_val
    )
)

dst_prediction_val = (
    X_dst_val[
        "lag_local_7d"
    ].to_numpy()
    +
    dst_delta_val
)

In [103]:
distance_val = (
    dst_meta_raw.loc[
        patch_val_index,
        "hours_to_dst"
    ]
    .abs()
    .to_numpy()
)

dst_grid_results = []

for width in [
    12,
    24,
    36,
    48
]:

    gate = (
        distance_val
        <= width
    )

    for lam in [
        0.0,
        0.25,
        0.50,
        0.75,
        1.0
    ]:

        pred = (
            patch_val_prediction.copy()
        )

        pred[
            gate
        ] = (
            (1 - lam)
            * patch_val_prediction[
                gate
            ]
            +
            lam
            * dst_prediction_val[
                gate
            ]
        )

        metrics = regression_metrics(
            y_patch_val,
            pred
        )

        dst_grid_results.append(
            {
                "window_h":
                    width,

                "lambda":
                    lam,

                "n":
                    gate.sum(),

                "RMSE":
                    metrics["RMSE"]
            }
        )

In [104]:
dst_grid_df = pd.DataFrame(
    dst_grid_results
)

print(
    dst_grid_df
    .sort_values("RMSE")
    .head(15)
)

    window_h  lambda    n        RMSE
6         24    0.25   97  196.423257
11        36    0.25  145  197.182458
7         24    0.50   97  197.784454
1         12    0.25   49  198.995911
2         12    0.50   49  199.433181
16        48    0.25  193  200.644192
10        36    0.00  145  201.315456
5         24    0.00   97  201.315456
15        48    0.00  193  201.315456
0         12    0.00   49  201.315456
3         12    0.75   49  202.609418
12        36    0.50  145  202.724661
8         24    0.75   97  205.274684
4         12    1.00   49  208.399424
17        48    0.50  193  217.103787


In [105]:
best_dst_row = (
    dst_grid_df
    .loc[
        dst_grid_df[
            "RMSE"
        ].idxmin()
    ]
)

best_dst_window = int(
    best_dst_row[
        "window_h"
    ]
)

best_dst_lambda = float(
    best_dst_row[
        "lambda"
    ]
)

print(
    "Best DST window :",
    best_dst_window,
    "h"
)

print(
    "Best DST lambda :",
    best_dst_lambda
)

print(
    "Best validation RMSE :",
    best_dst_row[
        "RMSE"
    ]
)

Best DST window : 24 h
Best DST lambda : 0.25
Best validation RMSE : 196.4232567932088


### Partie C — Analyse

Les lags classiques sont définis à partir d'un nombre fixe de
demi-heures :

$$
lag_{1jour}=48
$$

et :

$$
lag_{1semaine}=336.
$$

Autour des changements d'heure, ces lags ne correspondent cependant
plus nécessairement au même horaire local.

Le nouvel expert construit donc des références basées sur l'heure
locale française.

La différence :

$$
lag_{local}-lag_{UTC}
$$

permet explicitement au modèle de détecter les désalignements causés
par le changement d'heure.

Contrairement au précédent Event Expert, le DST Expert est entraîné
sur plusieurs années de changements d'heure historiques afin
d'augmenter fortement le nombre de transitions disponibles.

In [106]:
# ============================================================
# REFIT FINAL DST EXPERT — DONNEES < 2025
# Hyperparamètres déjà sélectionnés sur 2024
# ============================================================

BEST_DST_WINDOW = 24
BEST_DST_LAMBDA = 0.25

train_dst_final_mask = (
    dst_window_48
    &
    valid_dst
    &
    (
        dst_features.index
        < "2025-01-01"
    )
)

print(
    "Observations DST pour refit :",
    train_dst_final_mask.sum()
)

Observations DST pour refit : 4632


In [107]:
from sklearn.ensemble import HistGradientBoostingRegressor

dst_expert_final = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_iter=150,
    max_depth=3,
    min_samples_leaf=20,
    l2_regularization=10.0,
    early_stopping=True,
    random_state=42
)

dst_expert_final.fit(
    dst_features.loc[
        train_dst_final_mask
    ],
    dst_target.loc[
        train_dst_final_mask
    ]
)

print(
    "DST expert final entraîné."
)

DST expert final entraîné.


In [108]:
# ============================================================
# FEATURES DST 2025
# ============================================================

dst_X_2025 = (
    dst_features.loc[
        common_2025
    ]
)

dst_distance_2025 = (
    dst_meta_raw.loc[
        common_2025,
        "hours_to_dst"
    ]
    .abs()
    .to_numpy()
)

dst_gate_2025 = (
    dst_distance_2025
    <= BEST_DST_WINDOW
)

print(
    "Observations DST 2025 :",
    dst_gate_2025.sum()
)

Observations DST 2025 : 194


In [109]:
dst_delta_2025 = (
    dst_expert_final.predict(
        dst_X_2025
    )
)

dst_prediction_2025 = (
    dst_X_2025[
        "lag_local_7d"
    ].to_numpy()
    +
    dst_delta_2025
)

In [110]:
pred_patch_dst_2025 = (
    patch_common.copy()
)

pred_patch_dst_2025[
    dst_gate_2025
] = (
    (1 - BEST_DST_LAMBDA)
    * patch_common[
        dst_gate_2025
    ]
    +
    BEST_DST_LAMBDA
    * dst_prediction_2025[
        dst_gate_2025
    ]
)

In [111]:
dst_patch_metrics_2025 = regression_metrics(
    y_common_2025[
        dst_gate_2025
    ],
    patch_common[
        dst_gate_2025
    ]
)

dst_expert_metrics_2025 = regression_metrics(
    y_common_2025[
        dst_gate_2025
    ],
    pred_patch_dst_2025[
        dst_gate_2025
    ]
)

print(
    "Patch DST :",
    dst_patch_metrics_2025
)

print(
    "\nPatch + DST Expert :",
    dst_expert_metrics_2025
)

Patch DST : {'MSE': 396917.6790693806, 'RMSE': np.float64(630.0140308512031), 'MAE': 383.17351051886635, 'MAPE_%': np.float64(0.8195457689258876), 'R2': 0.9699822729126892}

Patch + DST Expert : {'MSE': 406372.4915283443, 'RMSE': np.float64(637.4735222174677), 'MAE': 463.6115571092267, 'MAPE_%': np.float64(0.9758099627801204), 'R2': 0.9692672330063784}


In [112]:
dst_gain_2025 = (
    100
    * (
        dst_patch_metrics_2025["RMSE"]
        - dst_expert_metrics_2025["RMSE"]
    )
    / dst_patch_metrics_2025["RMSE"]
)

print(
    "\nGain RMSE DST 2025 :",
    dst_gain_2025,
    "%"
)


Gain RMSE DST 2025 : -1.1840198790789154 %


In [113]:
dst_global_metrics = regression_metrics(
    y_common_2025,
    pred_patch_dst_2025
)

dst_global_gain = (
    100
    * (
        patch_common_metrics["RMSE"]
        - dst_global_metrics["RMSE"]
    )
    / patch_common_metrics["RMSE"]
)

print(
    "Patch global :",
    patch_common_metrics
)

print(
    "\nPatch + DST Expert :",
    dst_global_metrics
)

print(
    "\nGain global :",
    dst_global_gain,
    "%"
)

Patch global : {'MSE': 31535.528930303284, 'RMSE': np.float64(177.58245670759058), 'MAE': 128.24532895554032, 'MAPE_%': np.float64(0.2586217469956298), 'R2': 0.9997203651845527}

Patch + DST Expert : {'MSE': 31640.24055997954, 'RMSE': np.float64(177.87703775355473), 'MAE': 129.1361767627293, 'MAPE_%': np.float64(0.26035236602901), 'R2': 0.9997194366757173}

Gain global : -0.16588409205826643 %


In [114]:
pred_final_event_dst = (
    pred_refined_2025.copy()
)

In [115]:
pred_final_event_dst[
    dst_gate_2025
] = (
    (1 - BEST_DST_LAMBDA)
    * patch_common[
        dst_gate_2025
    ]
    +
    BEST_DST_LAMBDA
    * dst_prediction_2025[
        dst_gate_2025
    ]
)

In [116]:
final_event_dst_metrics = regression_metrics(
    y_common_2025,
    pred_final_event_dst
)

print(
    "Patch :",
    patch_common_metrics
)

print(
    "\nRefined Event Expert :",
    refined_metrics
)

print(
    "\nEvent + DST Experts :",
    final_event_dst_metrics
)

Patch : {'MSE': 31535.528930303284, 'RMSE': np.float64(177.58245670759058), 'MAE': 128.24532895554032, 'MAPE_%': np.float64(0.2586217469956298), 'R2': 0.9997203651845527}

Refined Event Expert : {'MSE': 31308.66995957857, 'RMSE': np.float64(176.94256118746154), 'MAE': 127.8516195648178, 'MAPE_%': np.float64(0.25761173413837657), 'R2': 0.9997223768098071}

Event + DST Experts : {'MSE': 31413.381589254826, 'RMSE': np.float64(177.2382057832194), 'MAE': 128.74246737200681, 'MAPE_%': np.float64(0.25934235317175675), 'R2': 0.9997214483009718}


In [117]:
gain_vs_patch = (
    100
    * (
        patch_common_metrics["RMSE"]
        - final_event_dst_metrics["RMSE"]
    )
    / patch_common_metrics["RMSE"]
)

gain_vs_event = (
    100
    * (
        refined_metrics["RMSE"]
        - final_event_dst_metrics["RMSE"]
    )
    / refined_metrics["RMSE"]
)

print(
    "\nGain vs Patch :",
    gain_vs_patch,
    "%"
)

print(
    "Gain supplémentaire vs Event Expert :",
    gain_vs_event,
    "%"
)


Gain vs Patch : 0.19385412881072414 %
Gain supplémentaire vs Event Expert : -0.16708506634796785 %


In [118]:
joblib.dump(
    {
        "dst_window_hours":
            BEST_DST_WINDOW,

        "dst_lambda":
            BEST_DST_LAMBDA,

        "dst_metrics_2025":
            dst_expert_metrics_2025,

        "dst_gain_2025":
            dst_gain_2025,

        "combined_metrics_2025":
            final_event_dst_metrics,

        "combined_prediction_2025":
            pred_final_event_dst,

        "index_2025":
            common_2025
    },
    "models/block7_event_dst_moe_results.joblib"
)

joblib.dump(
    dst_expert_final,
    "models/block7_dst_expert.joblib"
)

['models/block7_dst_expert.joblib']

### Conclusion — DST-Aware Expert

La correction DST fondée sur des lags alignés en heure locale
améliore nettement la validation temporelle.

Sur Q4 2024, le RMSE passe de :

$$
201.32\text{ MW}
$$

à :

$$
196.42\text{ MW}
$$

avec une fenêtre de :

$$
\pm24h
$$

autour du changement d'heure et un poids :

$$
\lambda=0.25.
$$

Cela représente une amélioration globale d'environ :

$$
2.43\%
$$

sur la période de validation.

Le résultat confirme que les erreurs observées pendant DST sont
en partie dues à un désalignement entre les lags calculés en UTC
et les rythmes de consommation exprimés en heure locale.

Le modèle DST est conservé comme expert spécialisé et doit maintenant
être évalué sur 2025 avec les paramètres entièrement figés.

### Conclusion — DST-Aware Expert

L'utilisation de lags alignés en heure locale produit une forte
amélioration sur la validation Q4 2024.

Cependant, cette amélioration ne se généralise pas à 2025.

Sur les périodes DST de 2025, le RMSE passe de :

$$
630.01\text{ MW}
$$

à :

$$
637.47\text{ MW}
$$

soit une dégradation d'environ :

$$
1.18\%
$$

Le modèle est donc rejeté dans sa forme actuelle.

Une limitation importante du protocole est que la sélection des
hyperparamètres a été réalisée sur Q4 2024, qui ne contient qu'une
transition DST, celle d'automne.

Les transitions de printemps et d'automne constituent deux régimes
différents et ne doivent probablement pas être modélisées avec le
même correcteur.

Le meilleur système reste donc le Refined Event Expert avec :

$$
RMSE = 176.94\text{ MW}.
$$

In [119]:
# ============================================================
# DIAGNOSTIC DST : PRINTEMPS VS AUTOMNE
# 2025 uniquement pour comprendre, PAS pour régler le modèle
# ============================================================

dst_spring_mask = (
    dst_X_2025[
        "dst_spring"
    ].astype(bool).to_numpy()
    &
    dst_gate_2025
)

dst_fall_mask = (
    dst_X_2025[
        "dst_fall"
    ].astype(bool).to_numpy()
    &
    dst_gate_2025
)


def compare_dst_regime(
    name,
    mask
):

    patch_m = regression_metrics(
        y_common_2025[mask],
        patch_common[mask]
    )

    expert_m = regression_metrics(
        y_common_2025[mask],
        pred_patch_dst_2025[mask]
    )

    gain = (
        100
        * (
            patch_m["RMSE"]
            - expert_m["RMSE"]
        )
        / patch_m["RMSE"]
    )

    return {
        "Regime": name,
        "n": mask.sum(),
        "Patch_RMSE": patch_m["RMSE"],
        "DST_Expert_RMSE": expert_m["RMSE"],
        "Gain_%": gain,
        "Patch_MAE": patch_m["MAE"],
        "DST_Expert_MAE": expert_m["MAE"]
    }


dst_direction_analysis = pd.DataFrame([
    compare_dst_regime(
        "Spring",
        dst_spring_mask
    ),

    compare_dst_regime(
        "Fall",
        dst_fall_mask
    )
])

print(
    dst_direction_analysis.round(4)
)

   Regime   n  Patch_RMSE  DST_Expert_RMSE  Gain_%  Patch_MAE  DST_Expert_MAE
0  Spring  97    615.3106         630.9225 -2.5372   379.3402        506.4800
1    Fall  97    644.3821         643.9579  0.0658   387.0068        420.7431


### Conclusion — Diagnostic Spring / Fall

La séparation des transitions DST confirme que l'expert fondé sur
les lags en heure locale ne généralise pas correctement.

Au printemps 2025 :

$$
RMSE : 615.31 \rightarrow 630.92\text{ MW}
$$

soit une dégradation de :

$$
2.54\%.
$$

En automne, le RMSE reste pratiquement inchangé, tandis que le MAE
se dégrade.

Le DST Expert est donc rejeté.

Les lags alignés sur l'heure locale restent une idée pertinente,
mais ils ne suffisent pas seuls à modéliser les changements de
comportement observés autour des transitions DST.

Le meilleur système reste le Refined Event Expert :

$$
RMSE_{2025}=176.94\text{ MW}.
$$

## Étape 7 — Concept Drift & Adaptive Learning

### Partie A — Mise en pratique

In [120]:
# ============================================================
# RESIDUS DU MEILLEUR MODELE 2025
# ============================================================

best_pred_2025 = np.asarray(
    pred_refined_2025
)

best_y_2025 = np.asarray(
    y_common_2025
)

best_residual_2025 = (
    best_y_2025
    -
    best_pred_2025
)

error_df = pd.DataFrame(
    {
        "y":
            best_y_2025,

        "prediction":
            best_pred_2025,

        "residual":
            best_residual_2025,

        "abs_error":
            np.abs(
                best_residual_2025
            ),

        "squared_error":
            best_residual_2025 ** 2
    },

    index=common_2025
)

print(
    error_df.head()
)

                                 y    prediction    residual   abs_error  \
2025-01-01 00:00:00+00:00  62487.0  62494.042538   -7.042538    7.042538   
2025-01-01 00:30:00+00:00  62236.0  62207.961674   28.038326   28.038326   
2025-01-01 01:00:00+00:00  61797.0  61791.439913    5.560087    5.560087   
2025-01-01 01:30:00+00:00  61762.0  61540.821520  221.178480  221.178480   
2025-01-01 02:00:00+00:00  60405.0  60491.284256  -86.284256   86.284256   

                           squared_error  
2025-01-01 00:00:00+00:00      49.597342  
2025-01-01 00:30:00+00:00     786.147727  
2025-01-01 01:00:00+00:00      30.914567  
2025-01-01 01:30:00+00:00   48919.920008  
2025-01-01 02:00:00+00:00    7444.972825  


In [121]:
monthly_drift = (
    error_df
    .resample("ME")
    .agg(
        n=("residual", "size"),

        bias=(
            "residual",
            "mean"
        ),

        MAE=(
            "abs_error",
            "mean"
        ),

        MSE=(
            "squared_error",
            "mean"
        )
    )
)

monthly_drift[
    "RMSE"
] = np.sqrt(
    monthly_drift[
        "MSE"
    ]
)

monthly_drift[
    "month"
] = (
    monthly_drift.index.month
)

print(
    monthly_drift[
        [
            "n",
            "bias",
            "MAE",
            "RMSE"
        ]
    ].round(3)
)

                              n    bias      MAE     RMSE
2025-01-31 00:00:00+00:00  1488  -4.163  145.597  195.374
2025-02-28 00:00:00+00:00  1344 -13.213  127.095  165.276
2025-03-31 00:00:00+00:00  1488  -9.231  150.680  238.602
2025-04-30 00:00:00+00:00  1440  -2.515  148.037  192.270
2025-05-31 00:00:00+00:00  1488   2.968  118.404  153.233
2025-06-30 00:00:00+00:00  1440   4.264  110.985  142.020
2025-07-31 00:00:00+00:00  1488  -7.791  102.917  130.325
2025-08-31 00:00:00+00:00  1488  -1.629   94.687  119.623
2025-09-30 00:00:00+00:00  1440  -1.756  118.345  151.163
2025-10-31 00:00:00+00:00  1488  -0.503  149.737  241.454
2025-11-30 00:00:00+00:00  1440   7.810  137.808  175.017
2025-12-31 00:00:00+00:00  1485  -1.743  129.981  169.498


In [122]:
# ============================================================
# RMSE GLISSANT
# ============================================================

error_df[
    "RMSE_7d"
] = np.sqrt(
    error_df[
        "squared_error"
    ]
    .rolling(
        336,
        min_periods=168
    )
    .mean()
)

error_df[
    "RMSE_30d"
] = np.sqrt(
    error_df[
        "squared_error"
    ]
    .rolling(
        1440,
        min_periods=720
    )
    .mean()
)

error_df[
    "MAE_30d"
] = (
    error_df[
        "abs_error"
    ]
    .rolling(
        1440,
        min_periods=720
    )
    .mean()
)

In [123]:
print(
    "RMSE 30 jours min :",
    error_df["RMSE_30d"].min()
)

print(
    "RMSE 30 jours médian :",
    error_df["RMSE_30d"].median()
)

print(
    "RMSE 30 jours max :",
    error_df["RMSE_30d"].max()
)

RMSE 30 jours min : 114.94280939350705
RMSE 30 jours médian : 165.31757372128573
RMSE 30 jours max : 260.8520495998831


In [124]:
worst_periods = (
    error_df[
        "RMSE_30d"
    ]
    .dropna()
    .sort_values(
        ascending=False
    )
    .head(10)
)

print(
    worst_periods
)

2025-11-24 11:00:00+00:00    260.852050
2025-11-24 10:00:00+00:00    260.841011
2025-11-24 10:30:00+00:00    260.820772
2025-11-24 09:30:00+00:00    260.778118
2025-11-24 09:00:00+00:00    260.759963
2025-11-24 11:30:00+00:00    260.721603
2025-11-24 08:30:00+00:00    260.699413
2025-11-24 17:00:00+00:00    260.684298
2025-11-24 13:00:00+00:00    260.681080
2025-11-24 16:30:00+00:00    260.680044
Name: RMSE_30d, dtype: float64


In [125]:
reference_mask = (
    error_df.index
    <
    "2025-04-01"
)

reference_mae = (
    error_df.loc[
        reference_mask,
        "abs_error"
    ]
    .mean()
)

reference_rmse = np.sqrt(
    error_df.loc[
        reference_mask,
        "squared_error"
    ]
    .mean()
)

print(
    "Reference MAE :",
    reference_mae
)

print(
    "Reference RMSE :",
    reference_rmse
)

Reference MAE : 141.59181262034056
Reference RMSE : 203.1146322932623


In [126]:
monthly_drift[
    "RMSE_ratio"
] = (
    monthly_drift[
        "RMSE"
    ]
    /
    reference_rmse
)

monthly_drift[
    "MAE_ratio"
] = (
    monthly_drift[
        "MAE"
    ]
    /
    reference_mae
)

print(
    monthly_drift[
        [
            "RMSE",
            "RMSE_ratio",
            "MAE",
            "MAE_ratio"
        ]
    ].round(3)
)

                              RMSE  RMSE_ratio      MAE  MAE_ratio
2025-01-31 00:00:00+00:00  195.374       0.962  145.597      1.028
2025-02-28 00:00:00+00:00  165.276       0.814  127.095      0.898
2025-03-31 00:00:00+00:00  238.602       1.175  150.680      1.064
2025-04-30 00:00:00+00:00  192.270       0.947  148.037      1.046
2025-05-31 00:00:00+00:00  153.233       0.754  118.404      0.836
2025-06-30 00:00:00+00:00  142.020       0.699  110.985      0.784
2025-07-31 00:00:00+00:00  130.325       0.642  102.917      0.727
2025-08-31 00:00:00+00:00  119.623       0.589   94.687      0.669
2025-09-30 00:00:00+00:00  151.163       0.744  118.345      0.836
2025-10-31 00:00:00+00:00  241.454       1.189  149.737      1.058
2025-11-30 00:00:00+00:00  175.017       0.862  137.808      0.973
2025-12-31 00:00:00+00:00  169.498       0.834  129.981      0.918


In [127]:
joblib.dump(
    {
        "error_df":
            error_df,

        "monthly_drift":
            monthly_drift,

        "reference_rmse":
            reference_rmse,

        "reference_mae":
            reference_mae
    },

    "models/block7_concept_drift_audit.joblib"
)

print(
    "Concept drift audit sauvegardé."
)

Concept drift audit sauvegardé.


In [128]:
# ============================================================
# REGIMES CONNUS A EXCLURE DU DIAGNOSTIC DE DRIFT
# ============================================================

dst_48h_mask = (
    dst_distance_2025
    <= 48
)

known_shock_mask = (
    event_mask_2025
    |
    dst_48h_mask
)

normal_drift_mask = (
    ~known_shock_mask
)

print(
    "Observations totales :",
    len(normal_drift_mask)
)

print(
    "Observations normales :",
    normal_drift_mask.sum()
)

print(
    "Observations exclues :",
    known_shock_mask.sum()
)

print(
    "Part exclue :",
    100 * known_shock_mask.mean(),
    "%"
)

Observations totales : 17517
Observations normales : 15310
Observations exclues : 2207
Part exclue : 12.599189358908488 %


In [129]:
error_df[
    "known_shock"
] = known_shock_mask

error_df[
    "normal_regime"
] = normal_drift_mask

In [130]:
normal_error_df = (
    error_df.loc[
        normal_drift_mask
    ]
    .copy()
)

In [131]:
monthly_normal = (
    normal_error_df
    .resample("ME")
    .agg(
        n=(
            "residual",
            "size"
        ),

        bias=(
            "residual",
            "mean"
        ),

        MAE=(
            "abs_error",
            "mean"
        ),

        MSE=(
            "squared_error",
            "mean"
        )
    )
)

monthly_normal[
    "RMSE"
] = np.sqrt(
    monthly_normal[
        "MSE"
    ]
)

print(
    monthly_normal[
        [
            "n",
            "bias",
            "MAE",
            "RMSE"
        ]
    ].round(3)
)

                              n    bias      MAE     RMSE
2025-01-31 00:00:00+00:00  1394  -1.137  143.620  193.047
2025-02-28 00:00:00+00:00  1344 -13.213  127.095  165.276
2025-03-31 00:00:00+00:00  1298  -6.690  126.367  166.192
2025-04-30 00:00:00+00:00  1241   0.411  149.239  193.968
2025-05-31 00:00:00+00:00  1108   9.913  113.479  146.761
2025-06-30 00:00:00+00:00  1296   4.006  109.377  140.399
2025-07-31 00:00:00+00:00  1344  -8.303  101.528  128.611
2025-08-31 00:00:00+00:00  1344  -1.839   93.780  117.950
2025-09-30 00:00:00+00:00  1440  -1.756  118.345  151.163
2025-10-31 00:00:00+00:00  1245   0.380  124.019  166.289
2025-11-30 00:00:00+00:00  1202   9.719  132.620  165.532
2025-12-31 00:00:00+00:00  1054   0.428  117.899  149.965


In [132]:
clean_squared_error = (
    error_df[
        "squared_error"
    ]
    .where(
        ~known_shock_mask
    )
)

clean_abs_error = (
    error_df[
        "abs_error"
    ]
    .where(
        ~known_shock_mask
    )
)

In [133]:
error_df[
    "RMSE_30d_clean"
] = np.sqrt(
    clean_squared_error
    .rolling(
        "30D",
        min_periods=720
    )
    .mean()
)

error_df[
    "MAE_30d_clean"
] = (
    clean_abs_error
    .rolling(
        "30D",
        min_periods=720
    )
    .mean()
)

In [134]:
print(
    "Clean RMSE 30d min :",
    error_df[
        "RMSE_30d_clean"
    ].min()
)

print(
    "Clean RMSE 30d median :",
    error_df[
        "RMSE_30d_clean"
    ].median()
)

print(
    "Clean RMSE 30d max :",
    error_df[
        "RMSE_30d_clean"
    ].max()
)

Clean RMSE 30d min : 112.59517600638597
Clean RMSE 30d median : 158.92646105699677
Clean RMSE 30d max : 202.91011332673165


In [135]:
worst_clean_periods = (
    error_df[
        "RMSE_30d_clean"
    ]
    .dropna()
    .sort_values(
        ascending=False
    )
    .head(10)
)

print(
    worst_clean_periods
)

2025-01-17 22:30:00+00:00    202.910113
2025-01-17 23:00:00+00:00    202.770956
2025-01-17 23:30:00+00:00    202.661776
2025-01-18 13:00:00+00:00    202.564304
2025-01-18 07:00:00+00:00    202.558118
2025-01-18 00:00:00+00:00    202.521727
2025-01-18 18:30:00+00:00    202.485297
2025-01-18 14:00:00+00:00    202.481153
2025-01-18 13:30:00+00:00    202.451041
2025-01-18 19:30:00+00:00    202.448114
Name: RMSE_30d_clean, dtype: float64


In [136]:
clean_reference_mask = (
    (error_df.index >= "2025-04-01")
    &
    (error_df.index < "2025-10-01")
    &
    normal_drift_mask
)

clean_reference_rmse = np.sqrt(
    error_df.loc[
        clean_reference_mask,
        "squared_error"
    ].mean()
)

clean_reference_mae = (
    error_df.loc[
        clean_reference_mask,
        "abs_error"
    ].mean()
)

print(
    "Clean reference RMSE :",
    clean_reference_rmse
)

print(
    "Clean reference MAE :",
    clean_reference_mae
)

Clean reference RMSE : 147.85905179615236
Clean reference MAE : 113.93336729063626


In [137]:
monthly_normal[
    "RMSE_ratio"
] = (
    monthly_normal[
        "RMSE"
    ]
    /
    clean_reference_rmse
)

monthly_normal[
    "MAE_ratio"
] = (
    monthly_normal[
        "MAE"
    ]
    /
    clean_reference_mae
)

print(
    monthly_normal[
        [
            "RMSE",
            "RMSE_ratio",
            "MAE",
            "MAE_ratio"
        ]
    ].round(3)
)

                              RMSE  RMSE_ratio      MAE  MAE_ratio
2025-01-31 00:00:00+00:00  193.047       1.306  143.620      1.261
2025-02-28 00:00:00+00:00  165.276       1.118  127.095      1.116
2025-03-31 00:00:00+00:00  166.192       1.124  126.367      1.109
2025-04-30 00:00:00+00:00  193.968       1.312  149.239      1.310
2025-05-31 00:00:00+00:00  146.761       0.993  113.479      0.996
2025-06-30 00:00:00+00:00  140.399       0.950  109.377      0.960
2025-07-31 00:00:00+00:00  128.611       0.870  101.528      0.891
2025-08-31 00:00:00+00:00  117.950       0.798   93.780      0.823
2025-09-30 00:00:00+00:00  151.163       1.022  118.345      1.039
2025-10-31 00:00:00+00:00  166.289       1.125  124.019      1.089
2025-11-30 00:00:00+00:00  165.532       1.120  132.620      1.164
2025-12-31 00:00:00+00:00  149.965       1.014  117.899      1.035


In [138]:
from sklearn.linear_model import LinearRegression

trend_df = (
    monthly_normal[
        ["RMSE"]
    ]
    .dropna()
    .copy()
)

trend_df[
    "t"
] = np.arange(
    len(trend_df)
)

trend_model = LinearRegression()

trend_model.fit(
    trend_df[
        ["t"]
    ],
    trend_df[
        "RMSE"
    ]
)

rmse_monthly_slope = (
    trend_model.coef_[0]
)

print(
    "Pente RMSE mensuelle :",
    rmse_monthly_slope,
    "MW / mois"
)

Pente RMSE mensuelle : -2.738319089309617 MW / mois


In [139]:
joblib.dump(
    {
        "monthly_normal":
            monthly_normal,

        "rmse_30d_clean":
            error_df[
                "RMSE_30d_clean"
            ],

        "mae_30d_clean":
            error_df[
                "MAE_30d_clean"
            ],

        "clean_reference_rmse":
            clean_reference_rmse,

        "clean_reference_mae":
            clean_reference_mae,

        "monthly_rmse_slope":
            rmse_monthly_slope,

        "known_shock_mask":
            known_shock_mask
    },

    "models/block7_clean_drift_audit.joblib"
)

print(
    "Clean drift audit sauvegardé."
)

Clean drift audit sauvegardé.


### Partie C — Analyse et améliorations guidées par la théorie

L'analyse initiale ne montre pas de dégradation monotone de la
performance au cours de 2025.

Les principales augmentations du RMSE apparaissent en mars et en
octobre, périodes correspondant aux transitions DST déjà identifiées
comme particulièrement difficiles.

Le maximum du RMSE glissant observé fin novembre peut également être
expliqué par la présence du changement d'heure du 26 octobre dans la
fenêtre glissante de 30 jours.

La référence janvier-mars est par ailleurs contaminée par la
transition DST de mars et surestime donc le niveau d'erreur normal.

Avant d'utiliser un détecteur automatique de concept drift, les
événements calendaires connus sont donc retirés du diagnostic.

L'objectif est de distinguer :

$$
\text{choc temporaire}
$$

de :

$$
\text{concept drift persistant}.
$$

### Conclusion — Concept Drift Audit

Après exclusion des événements calendaires connus et des périodes
proches des transitions DST, aucune dégradation monotone persistante
de la performance n'est observée en 2025.

Le RMSE glissant sur 30 jours passe de valeurs comprises entre :

$$
112.60 \text{ MW}
$$

et :

$$
202.91 \text{ MW}.
$$

Le maximum précédemment observé fin novembre disparaît après
suppression des périodes DST, ce qui indique qu'il était principalement
lié au changement d'heure d'octobre plutôt qu'à un concept drift.

La pente du RMSE mensuel est :

$$
-2.74 \text{ MW/mois}.
$$

Elle ne suggère donc pas de détérioration progressive du modèle.

Les variations observées semblent principalement dues à la
saisonnalité et à des régimes spécifiques plutôt qu'à une dérive
structurelle continue.

Un réentraînement permanent du modèle n'est donc pas justifié à ce
stade.

In [140]:
def causal_ewma_correction(
    y_true,
    y_pred,
    alpha,
    update_mask=None
):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    n = len(y_true)

    if update_mask is None:
        update_mask = np.ones(
            n,
            dtype=bool
        )

    corrected = np.empty(
        n,
        dtype=float
    )

    bias_history = np.empty(
        n,
        dtype=float
    )

    bias = 0.0

    for i in range(n):

        # ----------------------------
        # Prediction avec passé uniquement
        # ----------------------------

        corrected[i] = (
            y_pred[i]
            +
            bias
        )

        bias_history[i] = bias

        # ----------------------------
        # Résidu disponible APRES prediction
        # ----------------------------

        residual = (
            y_true[i]
            -
            y_pred[i]
        )

        # On évite que les événements rares
        # contaminent l'estimation du biais
        if update_mask[i]:

            bias = (
                (1 - alpha)
                * bias
                +
                alpha
                * residual
            )

    return (
        corrected,
        bias_history
    )

In [141]:
idx_adapt_val = (
    state[
        "val_index_2024"
    ]
)

y_adapt_val = np.asarray(
    state[
        "y_2024"
    ]
)

pred_adapt_val = np.asarray(
    state[
        "pred_patch_2024"
    ]
)

In [142]:
event_val = (
    event_X_2024
    .loc[
        idx_adapt_val,
        event_columns
    ]
    .max(axis=1)
    .astype(bool)
    .to_numpy()
)

dst_distance_val = (
    dst_meta_raw
    .loc[
        idx_adapt_val,
        "hours_to_dst"
    ]
    .abs()
    .to_numpy()
)

shock_val = (
    event_val
    |
    (
        dst_distance_val <= 48
    )
)

clean_update_val = (
    ~shock_val
)

In [143]:
alpha_grid = [
    0.0,
    0.001,
    0.0025,
    0.005,
    0.01,
    0.02,
    0.05
]

ewma_results = []

for alpha in alpha_grid:

    pred_tmp, _ = (
        causal_ewma_correction(
            y_adapt_val,
            pred_adapt_val,
            alpha=alpha,
            update_mask=clean_update_val
        )
    )

    metrics = regression_metrics(
        y_adapt_val,
        pred_tmp
    )

    ewma_results.append(
        {
            "alpha":
                alpha,

            "RMSE":
                metrics["RMSE"],

            "MAE":
                metrics["MAE"],

            "MAPE_%":
                metrics["MAPE_%"]
        }
    )

ewma_df = pd.DataFrame(
    ewma_results
)

print(
    ewma_df
    .sort_values(
        "RMSE"
    )
)

    alpha        RMSE         MAE    MAPE_%
1  0.0010  201.292952  137.703813  0.262379
0  0.0000  201.315456  137.625987  0.262253
2  0.0025  201.389360  137.776861  0.262521
3  0.0050  201.496759  137.824428  0.262628
4  0.0100  201.621499  137.897015  0.262797
5  0.0200  201.859696  138.081966  0.263212
6  0.0500  202.750714  138.936002  0.264882


In [144]:
best_alpha = float(
    ewma_df.loc[
        ewma_df["RMSE"].idxmin(),
        "alpha"
    ]
)

print(
    "Best alpha :",
    best_alpha
)

Best alpha : 0.001


In [145]:
best_pred_2025 = np.asarray(
    pred_refined_2025
)

best_y_2025 = np.asarray(
    y_common_2025
)

In [146]:
clean_update_2025 = (
    ~known_shock_mask
)

In [147]:
pred_adaptive_2025, bias_ewma_2025 = (
    causal_ewma_correction(
        best_y_2025,
        best_pred_2025,
        alpha=best_alpha,
        update_mask=clean_update_2025
    )
)

In [148]:
adaptive_metrics = regression_metrics(
    best_y_2025,
    pred_adaptive_2025
)

baseline_adaptive_metrics = regression_metrics(
    best_y_2025,
    best_pred_2025
)

print(
    "Refined Event Expert :",
    baseline_adaptive_metrics
)

print(
    "\nRefined Event Expert + EWMA :",
    adaptive_metrics
)

Refined Event Expert : {'MSE': 31308.66995957857, 'RMSE': np.float64(176.94256118746154), 'MAE': 127.8516195648178, 'MAPE_%': np.float64(0.25761173413837657), 'R2': 0.9997223768098071}

Refined Event Expert + EWMA : {'MSE': 31305.450936999692, 'RMSE': np.float64(176.93346471767202), 'MAE': 127.79825347618865, 'MAPE_%': np.float64(0.2575202073602706), 'R2': 0.9997224053538276}


In [149]:
adaptive_gain = (
    100
    *
    (
        baseline_adaptive_metrics["RMSE"]
        -
        adaptive_metrics["RMSE"]
    )
    /
    baseline_adaptive_metrics["RMSE"]
)

print(
    "\nGain RMSE :",
    adaptive_gain,
    "%"
)


Gain RMSE : 0.005140916763307438 %


In [150]:
bias_series = pd.Series(
    bias_ewma_2025,
    index=common_2025
)

print(
    "Bias EWMA min :",
    bias_series.min()
)

print(
    "Bias EWMA median :",
    bias_series.median()
)

print(
    "Bias EWMA max :",
    bias_series.max()
)

Bias EWMA min : -15.162714128468554
Bias EWMA median : -0.699102079875345
Bias EWMA max : 12.543599136197475


In [151]:
print(
    bias_series
    .resample("ME")
    .mean()
    .round(3)
)

2025-01-31 00:00:00+00:00   -0.968
2025-02-28 00:00:00+00:00   -5.883
2025-03-31 00:00:00+00:00   -8.785
2025-04-30 00:00:00+00:00   -1.457
2025-05-31 00:00:00+00:00    1.305
2025-06-30 00:00:00+00:00    4.511
2025-07-31 00:00:00+00:00   -1.323
2025-08-31 00:00:00+00:00   -5.284
2025-09-30 00:00:00+00:00   -0.506
2025-10-31 00:00:00+00:00   -2.647
2025-11-30 00:00:00+00:00    6.154
2025-12-31 00:00:00+00:00    3.797
Freq: ME, dtype: float64


In [152]:
joblib.dump(
    {
        "best_alpha":
            best_alpha,

        "validation_table":
            ewma_df,

        "baseline_metrics_2025":
            baseline_adaptive_metrics,

        "adaptive_metrics_2025":
            adaptive_metrics,

        "gain_rmse_pct":
            adaptive_gain,

        "prediction_2025":
            pred_adaptive_2025,

        "bias_series_2025":
            bias_series,

        "index_2025":
            common_2025
    },

    "models/block7_ewma_adaptive_results.joblib"
)

print(
    "EWMA adaptive state sauvegardé."
)

EWMA adaptive state sauvegardé.


### Conclusion — Adaptive Learning avec EWMA

L'approche EWMA (*Exponentially Weighted Moving Average*) a été testée
comme mécanisme léger d'adaptation en ligne.

Le meilleur coefficient sélectionné sur la validation 2024 est :

$$
\alpha = 0.001.
$$

Sur 2025, le RMSE du Refined Event Expert passe de :

$$
176.9426\text{ MW}
$$

à :

$$
176.9335\text{ MW}.
$$

Cela correspond à un gain très faible d'environ :

$$
0.0051\%.
$$

Le MAE diminue également légèrement :

$$
127.852
\rightarrow
127.798\text{ MW}.
$$

La correction EWMA reste stable au cours de l'année, avec un biais
adaptatif compris approximativement entre :

$$
-15.16\text{ MW}
$$

et :

$$
+12.54\text{ MW}.
$$

La médiane du biais est proche de zéro :

$$
-0.70\text{ MW}.
$$

Le mécanisme adaptatif ne présente donc pas de dérive ou
d'instabilité importante.

Cependant, l'amélioration obtenue est extrêmement faible en pratique.
L'EWMA est donc conservée comme expérience d'Adaptive Learning, mais
son intégration dans le modèle final dépendra de la comparaison
statistique réalisée à l'étape suivante.

Ces résultats sont cohérents avec le diagnostic de concept drift :
aucune dérive structurelle importante n'ayant été observée en 2025,
une correction adaptative ne peut apporter qu'un gain limité.

## Étape 9 — Statistical Forecast Comparison

### Partie A — Mise en pratique

In [153]:
import numpy as np
import pandas as pd

from scipy.stats import norm


def dm_hac_test(
    y_true,
    pred_a,
    pred_b,
    hac_lag=48,
    loss="mse"
):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    pred_a = np.asarray(
        pred_a,
        dtype=float
    )

    pred_b = np.asarray(
        pred_b,
        dtype=float
    )

    error_a = (
        y_true
        - pred_a
    )

    error_b = (
        y_true
        - pred_b
    )

    # ==========================================
    # Fonction de perte
    # ==========================================

    if loss == "mse":

        loss_a = (
            error_a ** 2
        )

        loss_b = (
            error_b ** 2
        )

    elif loss == "mae":

        loss_a = np.abs(
            error_a
        )

        loss_b = np.abs(
            error_b
        )

    else:

        raise ValueError(
            "loss doit être 'mse' ou 'mae'"
        )


    # ==========================================
    # Différentiel de perte
    # positif => B meilleur
    # ==========================================

    d = (
        loss_a
        -
        loss_b
    )

    n = len(d)

    d_mean = (
        d.mean()
    )

    centered = (
        d
        -
        d_mean
    )


    # ==========================================
    # Variance longue période Newey-West
    # ==========================================

    gamma_0 = (
        np.dot(
            centered,
            centered
        )
        /
        n
    )

    long_run_var = (
        gamma_0
    )

    max_lag = min(
        hac_lag,
        n - 1
    )

    for lag in range(
        1,
        max_lag + 1
    ):

        gamma_k = (
            np.dot(
                centered[lag:],
                centered[:-lag]
            )
            /
            n
        )

        # Poids Bartlett / Newey-West
        weight = (
            1
            -
            lag
            /
            (
                max_lag
                + 1
            )
        )

        long_run_var += (
            2
            *
            weight
            *
            gamma_k
        )


    # Sécurité numérique
    if long_run_var <= 0:

        return {
            "DM_stat": np.nan,
            "p_value": np.nan,
            "mean_loss_diff": d_mean,
            "hac_lag": hac_lag,
            "loss": loss
        }


    variance_mean = (
        long_run_var
        /
        n
    )

    dm_stat = (
        d_mean
        /
        np.sqrt(
            variance_mean
        )
    )

    # Test bilatéral
    p_value = (
        2
        *
        norm.sf(
            abs(dm_stat)
        )
    )

    return {
        "DM_stat":
            dm_stat,

        "p_value":
            p_value,

        "mean_loss_diff":
            d_mean,

        "hac_lag":
            hac_lag,

        "loss":
            loss
    }

In [154]:
models_pred = {
    "Patch":
        patch_common,

    "Event Expert":
        pred_refined_2025,

    "Event + EWMA":
        pred_adaptive_2025
}

In [155]:
comparisons = [
    (
        "Patch",
        "Event Expert"
    ),

    (
        "Event Expert",
        "Event + EWMA"
    ),

    (
        "Patch",
        "Event + EWMA"
    )
]

In [156]:
dm_results = []

for model_a, model_b in comparisons:

    result = dm_hac_test(
        y_true=y_common_2025,

        pred_a=models_pred[
            model_a
        ],

        pred_b=models_pred[
            model_b
        ],

        hac_lag=48,

        loss="mse"
    )

    dm_results.append(
        {
            "Model_A":
                model_a,

            "Model_B":
                model_b,

            **result
        }
    )


dm_df_48 = pd.DataFrame(
    dm_results
)

print(
    dm_df_48.round(6)
)

        Model_A       Model_B   DM_stat   p_value  mean_loss_diff  hac_lag  \
0         Patch  Event Expert  3.197891  0.001384      226.858971       48   
1  Event Expert  Event + EWMA  0.201131  0.840596        3.219023       48   
2         Patch  Event + EWMA  3.173882  0.001504      230.077993       48   

  loss  
0  mse  
1  mse  
2  mse  


In [157]:
dm_results_week = []

for model_a, model_b in comparisons:

    result = dm_hac_test(
        y_true=y_common_2025,

        pred_a=models_pred[
            model_a
        ],

        pred_b=models_pred[
            model_b
        ],

        hac_lag=336,

        loss="mse"
    )

    dm_results_week.append(
        {
            "Model_A":
                model_a,

            "Model_B":
                model_b,

            **result
        }
    )


dm_df_336 = pd.DataFrame(
    dm_results_week
)

print(
    dm_df_336.round(6)
)

        Model_A       Model_B   DM_stat   p_value  mean_loss_diff  hac_lag  \
0         Patch  Event Expert  3.309755  0.000934      226.858971      336   
1  Event Expert  Event + EWMA  0.184398  0.853701        3.219023      336   
2         Patch  Event + EWMA  3.286241  0.001015      230.077993      336   

  loss  
0  mse  
1  mse  
2  mse  


In [158]:
dm_results_mae = []

for model_a, model_b in comparisons:

    result = dm_hac_test(
        y_true=y_common_2025,

        pred_a=models_pred[
            model_a
        ],

        pred_b=models_pred[
            model_b
        ],

        hac_lag=336,

        loss="mae"
    )

    dm_results_mae.append(
        {
            "Model_A":
                model_a,

            "Model_B":
                model_b,

            **result
        }
    )


dm_df_mae = pd.DataFrame(
    dm_results_mae
)

print(
    dm_df_mae.round(6)
)

        Model_A       Model_B   DM_stat   p_value  mean_loss_diff  hac_lag  \
0         Patch  Event Expert  3.414723  0.000638        0.393709      336   
1  Event Expert  Event + EWMA  1.147361  0.251233        0.053366      336   
2         Patch  Event + EWMA  3.616053  0.000299        0.447075      336   

  loss  
0  mae  
1  mae  
2  mae  


In [159]:
dm_summary = pd.concat(
    [
        dm_df_48.assign(
            Test="MSE — HAC 1 day"
        ),

        dm_df_336.assign(
            Test="MSE — HAC 1 week"
        ),

        dm_df_mae.assign(
            Test="MAE — HAC 1 week"
        )
    ],

    ignore_index=True
)

dm_summary = dm_summary[
    [
        "Test",
        "Model_A",
        "Model_B",
        "DM_stat",
        "p_value",
        "mean_loss_diff"
    ]
]

print(
    dm_summary.round(6)
)

               Test       Model_A       Model_B   DM_stat   p_value  \
0   MSE — HAC 1 day         Patch  Event Expert  3.197891  0.001384   
1   MSE — HAC 1 day  Event Expert  Event + EWMA  0.201131  0.840596   
2   MSE — HAC 1 day         Patch  Event + EWMA  3.173882  0.001504   
3  MSE — HAC 1 week         Patch  Event Expert  3.309755  0.000934   
4  MSE — HAC 1 week  Event Expert  Event + EWMA  0.184398  0.853701   
5  MSE — HAC 1 week         Patch  Event + EWMA  3.286241  0.001015   
6  MAE — HAC 1 week         Patch  Event Expert  3.414723  0.000638   
7  MAE — HAC 1 week  Event Expert  Event + EWMA  1.147361  0.251233   
8  MAE — HAC 1 week         Patch  Event + EWMA  3.616053  0.000299   

   mean_loss_diff  
0      226.858971  
1        3.219023  
2      230.077993  
3      226.858971  
4        3.219023  
5      230.077993  
6        0.393709  
7        0.053366  
8        0.447075  


In [160]:
joblib.dump(
    {
        "dm_1day_mse":
            dm_df_48,

        "dm_1week_mse":
            dm_df_336,

        "dm_1week_mae":
            dm_df_mae,

        "summary":
            dm_summary
    },

    "models/block7_statistical_comparison.joblib"
)

print(
    "Comparaisons statistiques sauvegardées."
)

Comparaisons statistiques sauvegardées.


### Partie C — Analyse et améliorations guidées par la théorie

Le test de Diebold-Mariano compare la précision prédictive de deux
prévisions sur les mêmes observations.

Le différentiel de perte est défini par :

$$
d_t =
L(e_{A,t}) -
L(e_{B,t}).
$$

Une moyenne positive indique que le modèle B présente une perte
moyenne inférieure au modèle A.

Une estimation HAC/Newey-West est utilisée afin de tenir compte de
l'autocorrélation potentielle du différentiel de perte.

Deux longueurs de dépendance sont étudiées :

$$
48 = 1 \text{ jour}
$$

et :

$$
336 = 1 \text{ semaine}.
$$

La perte quadratique permet d'étudier les différences de MSE/RMSE,
tandis que la perte absolue fournit une comparaison plus robuste aux
erreurs extrêmes.

La significativité statistique sera distinguée de l'importance
pratique de l'amélioration.

### Conclusion — Comparaison statistique des prévisions

Le test de Diebold-Mariano confirme que le Refined Event Expert
améliore significativement la précision prédictive du Patch Transformer.

Pour la perte quadratique, les comparaisons donnent :

$$
p = 0.00138
$$

avec une correction HAC sur un jour, et :

$$
p = 0.00093
$$

avec une correction HAC sur une semaine.

La comparaison fondée sur le MAE conduit également à :

$$
p = 0.00064.
$$

Les différentiels moyens de perte sont positifs, ce qui indique
que le Refined Event Expert présente une erreur moyenne inférieure
au Patch.

En revanche, l'ajout de la correction EWMA ne produit pas
d'amélioration statistiquement détectable par rapport au
Refined Event Expert :

$$
p \approx 0.85
$$

pour la perte quadratique et :

$$
p \approx 0.25
$$

pour la perte absolue.

L'amélioration apportée par l'EWMA est donc considérée comme
négligeable en pratique.

Le Refined Event Expert est retenu comme modèle principal du Bloc 7,
tandis que l'EWMA est conservée uniquement comme expérience
d'Adaptive Learning.

## Étape 10 — Sélection finale du Bloc 7

## Conclusion finale — Bloc 7

Le Bloc 7 a étudié l'amélioration du modèle séquentiel à travers
plusieurs stratégies : corrections calendaires globales, SARIMAX,
experts spécialisés, gestion du changement d'heure, adaptation
en ligne et comparaison statistique.

Les corrections globales Calendar Ridge et SARIMAX n'améliorent pas
le Patch Transformer et sont rejetées.

L'analyse des erreurs montre en revanche que certaines périodes
calendaires constituent des régimes spécifiques.

Un expert spécialisé activé uniquement sur les jours fériés,
les périodes de fin d'année et les lendemains de jours fériés
permet de réduire le RMSE de :

$$
177.58\text{ MW}
$$

à :

$$
176.94\text{ MW}.
$$

Cela représente une amélioration d'environ :

$$
0.36\%.
$$

Les tests de Diebold-Mariano confirment que cette amélioration est
statistiquement détectable pour les pertes MSE et MAE.

L'adaptation EWMA apporte une amélioration supplémentaire très faible :

$$
176.94 \rightarrow 176.93\text{ MW},
$$

mais cette différence n'est pas statistiquement significative.
L'EWMA est donc conservée comme expérience mais n'est pas intégrée
au modèle principal.

L'expert DST fondé sur les lags alignés en heure locale améliore
la validation 2024 mais ne généralise pas à 2025. Il est donc rejeté.

Le modèle final retenu pour le Bloc 7 est :

$$
\boxed{
\text{Ridge + MLP1 + Patch Transformer + Refined Event Expert}
}
$$

avec une performance 2025 de :

$$
\boxed{RMSE = 176.94\text{ MW}}.
$$

L'année 2025 reste une période de développement avancé.
Le véritable test final sur 2026 reste intact et ne sera utilisé
qu'après verrouillage complet du pipeline.

In [161]:
block7_final_state = {
    "model_name":
        "Ridge + MLP1 + Patch Transformer + Refined Event Expert",

    "index_2025":
        common_2025,

    "y_2025":
        y_common_2025,

    "prediction_2025":
        pred_refined_2025,

    "metrics_2025":
        refined_metrics,

    "rmse_2025":
        refined_metrics["RMSE"],

    "event_columns":
        refined_event_columns,

    "event_lambda":
        best_lambda,

    "ewma_selected":
        False,

    "ewma_alpha":
        best_alpha,

    "dst_expert_selected":
        False,

    "dm_summary":
        dm_summary
}

joblib.dump(
    block7_final_state,
    "models/block7_final_state.joblib"
)

print(
    "Bloc 7 final sauvegardé."
)

Bloc 7 final sauvegardé.
